# 05 – WVS Feature Engineering

## Research Objective

Prepare and organize relevant predictor variables for explaining and predicting life satisfaction (Q49) using the World Values Survey Wave 7 dataset.

## Main Tasks

1. Review candidate predictor variables.
2. Select relevant variables based on data quality, measurement type, and exploratory findings.
3. Identify redundant variables and alternative versions.
4. Define feature groups for comparative modeling.
5. Establish consistent rules for missing values and categorical encoding.
6. Prepare a modeling-ready feature selection framework.

## Target Variable

Q49 – Life Satisfaction (scale: 1–10).

Q49 must never be included as a predictor.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from pathlib import Path

# Notebook is expected to run from the notebooks/ directory
PROJECT_DIR = Path.cwd().parent

FEATURE_SELECTION_DIR = (
    PROJECT_DIR / "outputs" / "feature_selection"
)

DICTIONARY_FILE = (
    FEATURE_SELECTION_DIR
    / "WVS_Master_Variable_Dictionary.csv"
)

if not DICTIONARY_FILE.exists():
    raise FileNotFoundError(
        f"Master Variable Dictionary not found: {DICTIONARY_FILE}"
    )

master_dictionary = pd.read_csv(DICTIONARY_FILE)

print("Dictionary loaded successfully.")
print("Number of variables:", len(master_dictionary))
print("Number of columns:", len(master_dictionary.columns))

display(master_dictionary.head())

Dictionary loaded successfully.
Number of variables: 372
Number of columns: 18


,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses
0,Q1,Personal Values,Predictor candidate,Candidate,Personal Values & Wellbeing,41,int64,0,0.0,7,155,-1: 27; -2: 61; -5: 67,1 | 2 | 3 | 4 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.0,97065.0
1,Q2,Personal Values,Predictor candidate,Candidate,Personal Values & Wellbeing,42,int64,0,0.0,7,312,-1: 84; -2: 141; -5: 87,1 | 2 | 3 | 4 | -1 | -2 | -5,Survey question / question-derived field,Ordinal,66.0,100.0,96908.0
2,Q3,Personal Values,Predictor candidate,Candidate,Personal Values & Wellbeing,43,int64,0,0.0,7,535,-1: 202; -2: 196; -5: 137,1 | 2 | 3 | 4 | -1 | -2 | -5,Survey question / question-derived field,Ordinal,66.0,100.0,96685.0
3,Q4,Personal Values,Predictor candidate,Candidate,Personal Values & Wellbeing,44,int64,0,0.0,7,1090,-1: 587; -2: 323; -5: 180,3 | 4 | 2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.0,96130.0
4,Q5,Personal Values,Predictor candidate,Candidate,Personal Values & Wellbeing,45,int64,0,0.0,7,1160,-1: 368; -2: 289; -5: 503,1 | 3 | 2 | 4 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.0,96060.0


In [2]:
# Summarize variables by feature group and model status

feature_summary = (
    master_dictionary
    .groupby(
        ["Feature_group", "Model_status"],
        dropna=False
    )
    .size()
    .unstack(fill_value=0)
)

display(feature_summary)

Model_status,Candidate,Keep for exploratory analysis,Review before modeling,Target only; exclude from X
Feature_group,,,,
Basic Demographic Background,15,0,0,0
Child-rearing Values,11,0,0,0
Civic and Political Activism,4,0,0,0
Conventional Political Participation,4,0,0,0
Country-specific institutional confidence,17,0,0,0
Economic Security,7,0,0,0
Economic Values,6,0,0,0
Educational Background,12,0,0,0
Electoral Integrity and Political Voice,12,0,0,0


In [3]:
measurement_summary = (
    master_dictionary
    .groupby(
        ["Feature_group", "Measurement_type"]
    )
    .size()
    .unstack(fill_value=0)
)

display(measurement_summary)

Measurement_type,Binary,Nominal,Numeric,Ordinal
Feature_group,,,,
Basic Demographic Background,3,8,4,0
Child-rearing Values,11,0,0,0
Civic and Political Activism,0,0,0,4
Conventional Political Participation,0,0,0,4
Country-specific institutional confidence,0,0,0,17
Economic Security,0,1,0,6
Economic Values,0,1,0,5
Educational Background,0,4,0,8
Electoral Integrity and Political Voice,0,0,0,12


In [4]:
print("Main Categories:")
display(
    master_dictionary["Category"]
    .value_counts(dropna=False)
)

print("\nDetailed Feature Groups:")
display(
    master_dictionary["Feature_group"]
    .value_counts(dropna=False)
)

Main Categories:


Category
Political & Institutional      154
Social & Community              84
Personal Values & Wellbeing     55
Demographic & Background        36
Economic                        18
Security                        18
Science & Technology             6
Target Variable                  1
Name: count, dtype: int64


Detailed Feature Groups:


Feature_group
Institutional confidence                           25
Political Culture and Democratic Preferences       25
Ethical Values and Moral Norms                     20
Political Trust - Parliament, Government and UN    18
Country-specific institutional confidence          17
Family and Gender Norms                            15
Basic Demographic Background                       15
Political Trust and Government Attitudes           15
Organizational membership                          12
Recoded membership variables                       12
Electoral Integrity and Political Voice            12
Educational Background                             12
Child-rearing Values                               11
Personal and Neighborhood Security                 11
Immigration Attitudes                              10
Social Tolerance                                    9
Perceived Corruption                                9
News and Information Sources                        8
Work and Socia

In [5]:
category_summary = (
    master_dictionary
    .groupby(
        ["Category", "Model_status"],
        dropna=False
    )
    .size()
    .unstack(fill_value=0)
)

display(category_summary)

Model_status,Candidate,Keep for exploratory analysis,Review before modeling,Target only; exclude from X
Category,,,,
Demographic & Background,36,0,0,0
Economic,18,0,0,0
Personal Values & Wellbeing,55,0,0,0
Political & Institutional,153,1,0,0
Science & Technology,6,0,0,0
Security,18,0,0,0
Social & Community,81,0,3,0
Target Variable,0,0,0,1


In [6]:
category_measurement_summary = (
    master_dictionary[
        master_dictionary["Model_status"] == "Candidate"
    ]
    .groupby(
        ["Category", "Measurement_type"]
    )
    .size()
    .unstack(fill_value=0)
)

display(category_measurement_summary)

Measurement_type,Binary,Nominal,Numeric,Ordinal
Category,,,,
Demographic & Background,3,21,4,8
Economic,1,2,0,15
Personal Values & Wellbeing,4,10,0,41
Political & Institutional,3,10,0,140
Science & Technology,0,0,0,6
Security,5,0,0,13
Social & Community,33,8,0,40


## Feature Selection Strategy

Predictor candidates are selected based on:

1. Relevance to the research question.
2. Data quality and missingness.
3. Measurement type and appropriate encoding.
4. Redundancy with other variables.
5. Country coverage.
6. Exploratory findings and potential predictive value.

Variables are assigned one of four decisions:

- Keep: Include in the main candidate pool.
- Optional: Consider in extended models or sensitivity analysis.
- Review: Require additional inspection before modeling.
- Exclude: Do not include in the selected predictor pool.

The selection process aims to retain theoretically relevant and less intuitive predictors while avoiding redundant variables and data leakage.

In [7]:
feature_selection = master_dictionary.copy()

feature_selection["Selection_decision"] = "Review"
feature_selection["Selection_reason"] = ""

# Q49 is the target variable, not a predictor
feature_selection.loc[
    feature_selection["Variable"] == "Q49",
    "Selection_decision"
] = "Exclude"

feature_selection.loc[
    feature_selection["Variable"] == "Q49",
    "Selection_reason"
] = "Target variable; must not be included in predictors."

In [8]:
economic_keep = [
    "Q50", "Q51", "Q52", "Q53", "Q54", "Q55", "Q56",
    "Q106", "Q107", "Q108", "Q109", "Q110", "Q111",
    "Q285", "Q286", "Q287", "Q288"
]

economic_optional = ["Q288R"]

feature_selection.loc[
    feature_selection["Variable"].isin(economic_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(economic_keep),
    "Selection_reason"
] = "Economic predictor candidate; retain for subsequent data-quality and redundancy checks."

feature_selection.loc[
    feature_selection["Variable"].isin(economic_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(economic_optional),
    "Selection_reason"
] = "Alternative recoded version of Q288; do not use simultaneously with Q288."

display(
    feature_selection[
        feature_selection["Category"] == "Economic"
    ][
        [
            "Variable",
            "Feature_group",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Feature_group,Measurement_type,Selection_decision,Selection_reason
48,Q50,Economic Security,Ordinal,Keep,Economic predictor candidate; retain for subse...
49,Q51,Economic Security,Ordinal,Keep,Economic predictor candidate; retain for subse...
50,Q52,Economic Security,Ordinal,Keep,Economic predictor candidate; retain for subse...
51,Q53,Economic Security,Ordinal,Keep,Economic predictor candidate; retain for subse...
52,Q54,Economic Security,Ordinal,Keep,Economic predictor candidate; retain for subse...
53,Q55,Economic Security,Ordinal,Keep,Economic predictor candidate; retain for subse...
54,Q56,Economic Security,Nominal,Keep,Economic predictor candidate; retain for subse...
132,Q106,Economic Values,Ordinal,Keep,Economic predictor candidate; retain for subse...
133,Q107,Economic Values,Ordinal,Keep,Economic predictor candidate; retain for subse...
134,Q108,Economic Values,Ordinal,Keep,Economic predictor candidate; retain for subse...


In [9]:
feature_selection[
    feature_selection["Category"] == "Economic"
][
    [
        "Variable",
        "Feature_group",
        "Measurement_type",
        "Selection_decision",
        "Selection_reason"
    ]
]

,Variable,Feature_group,Measurement_type,Selection_decision,Selection_reason
48,Q50,Economic Security,Ordinal,Keep,Economic predictor candidate; retain for subse...
49,Q51,Economic Security,Ordinal,Keep,Economic predictor candidate; retain for subse...
50,Q52,Economic Security,Ordinal,Keep,Economic predictor candidate; retain for subse...
51,Q53,Economic Security,Ordinal,Keep,Economic predictor candidate; retain for subse...
52,Q54,Economic Security,Ordinal,Keep,Economic predictor candidate; retain for subse...
53,Q55,Economic Security,Ordinal,Keep,Economic predictor candidate; retain for subse...
54,Q56,Economic Security,Nominal,Keep,Economic predictor candidate; retain for subse...
132,Q106,Economic Values,Ordinal,Keep,Economic predictor candidate; retain for subse...
133,Q107,Economic Values,Ordinal,Keep,Economic predictor candidate; retain for subse...
134,Q108,Economic Values,Ordinal,Keep,Economic predictor candidate; retain for subse...


In [10]:
feature_selection["Selection_decision"].value_counts()

Selection_decision
Review      353
Keep         17
Optional      1
Exclude       1
Name: count, dtype: int64

In [11]:
demographic_review = feature_selection[
    feature_selection["Category"] == "Demographic & Background"
][
    [
        "Variable",
        "Feature_group",
        "Measurement_type",
        "Missing percentage (NaN)",
        "Countries_with_valid_responses",
        "Selection_decision"
    ]
]

demographic_review

,Variable,Feature_group,Measurement_type,Missing percentage (NaN),Countries_with_valid_responses,Selection_decision
290,Q260,Basic Demographic Background,Nominal,0.0,NaN,Review
291,Q261,Basic Demographic Background,Numeric,0.0,NaN,Review
292,Q262,Basic Demographic Background,Numeric,0.0,NaN,Review
293,Q263,Basic Demographic Background,Binary,0.0,NaN,Review
294,Q264,Basic Demographic Background,Binary,0.0,NaN,Review
295,Q265,Basic Demographic Background,Binary,0.0,NaN,Review
296,Q266,Basic Demographic Background,Nominal,0.0,NaN,Review
297,Q267,Basic Demographic Background,Nominal,0.0,NaN,Review
298,Q268,Basic Demographic Background,Nominal,0.0,NaN,Review
299,Q269,Basic Demographic Background,Nominal,0.0,NaN,Review


In [12]:
# Basic Demographic Background
demographic_keep = [
    "Q260", "Q262", "Q263",
    "Q270", "Q271", "Q273", "Q274"
]

demographic_optional = [
    "Q264", "Q265", "Q266", "Q267", "Q268"
]

demographic_review = ["Q269", "Q272"]

demographic_exclude = ["Q261"]


# Educational Background
education_keep = ["Q275", "Q277", "Q278"]

education_optional = [
    "Q275R", "Q276", "Q276R",
    "Q277R", "Q278R"
]

education_exclude = [
    "Q275A", "Q276A", "Q277A", "Q278A"
]


# Apply decisions
feature_selection.loc[
    feature_selection["Variable"].isin(demographic_keep + education_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(demographic_optional + education_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(demographic_review),
    "Selection_decision"
] = "Review"

feature_selection.loc[
    feature_selection["Variable"].isin(demographic_exclude + education_exclude),
    "Selection_decision"
] = "Exclude"

In [13]:
feature_selection.loc[
    feature_selection["Variable"].isin(demographic_keep),
    "Selection_reason"
] = "Relevant demographic background predictor."

feature_selection.loc[
    feature_selection["Variable"].isin(demographic_optional),
    "Selection_reason"
] = "Potentially useful family or migration background variable; review redundancy and coverage."

feature_selection.loc[
    feature_selection["Variable"].isin(demographic_exclude),
    "Selection_reason"
] = "Redundant with Q262; use respondent age Q262 instead."

feature_selection.loc[
    feature_selection["Variable"].isin(demographic_review),
    "Selection_reason"
] = "Requires further review of coding and cross-country comparability."

feature_selection.loc[
    feature_selection["Variable"].isin(education_keep),
    "Selection_reason"
] = "Relevant educational background predictor."

feature_selection.loc[
    feature_selection["Variable"].isin(education_optional),
    "Selection_reason"
] = "Alternative or family education predictor; review missingness and redundancy."

feature_selection.loc[
    feature_selection["Variable"].isin(education_exclude),
    "Selection_reason"
] = "Country-specific education coding; standard ISCED variables preferred."

In [14]:
feature_selection[
    feature_selection["Variable"].isin(
        ["Q279", "Q280", "Q281", "Q282", "Q283", "Q284"]
    )
][
    [
        "Variable",
        "Feature_group",
        "Measurement_type",
        "Model_status"
    ]
]

,Variable,Feature_group,Measurement_type,Model_status
317,Q279,Employment and Occupational Background,Nominal,Candidate
318,Q280,Employment and Occupational Background,Nominal,Candidate
319,Q281,Employment and Occupational Background,Nominal,Candidate
320,Q282,Employment and Occupational Background,Nominal,Candidate
321,Q283,Employment and Occupational Background,Nominal,Candidate
322,Q284,Employment and Occupational Background,Nominal,Candidate


In [15]:
employment_keep = [
    "Q279", "Q281", "Q283", "Q284"
]

employment_optional = [
    "Q280", "Q282"
]

feature_selection.loc[
    feature_selection["Variable"].isin(employment_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(employment_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(employment_keep),
    "Selection_reason"
] = "Relevant employment or occupational background predictor."

feature_selection.loc[
    feature_selection["Variable"].isin(employment_optional),
    "Selection_reason"
] = "Spouse-related employment predictor; relatively high missingness."

In [16]:
feature_selection[
    feature_selection["Variable"].isin(
        ["Q279", "Q280", "Q281", "Q282", "Q283", "Q284"]
    )
][
    [
        "Variable",
        "Measurement_type",
        "Selection_decision",
        "Selection_reason"
    ]
]

,Variable,Measurement_type,Selection_decision,Selection_reason
317,Q279,Nominal,Keep,Relevant employment or occupational background...
318,Q280,Nominal,Optional,Spouse-related employment predictor; relativel...
319,Q281,Nominal,Keep,Relevant employment or occupational background...
320,Q282,Nominal,Optional,Spouse-related employment predictor; relativel...
321,Q283,Nominal,Keep,Relevant employment or occupational background...
322,Q284,Nominal,Keep,Relevant employment or occupational background...


In [17]:
master_dictionary.loc[
    master_dictionary["Variable"] == "Q284",
    "Measurement_type"
] = "Nominal"

In [18]:

feature_selection.loc[
    feature_selection["Variable"] == "Q284",
    "Measurement_type"
] = "Nominal"

In [19]:
feature_selection[
    feature_selection["Variable"].isin(
        ["Q279", "Q280", "Q281", "Q282", "Q283", "Q284"]
    )
][
    ["Variable", "Measurement_type", "Selection_decision"]
]

,Variable,Measurement_type,Selection_decision
317,Q279,Nominal,Keep
318,Q280,Nominal,Optional
319,Q281,Nominal,Keep
320,Q282,Nominal,Optional
321,Q283,Nominal,Keep
322,Q284,Nominal,Keep


In [20]:
from pathlib import Path

# Project output directory
PROJECT_DIR = Path.cwd().parent
output_dir = PROJECT_DIR / "outputs" / "feature_selection"
output_dir.mkdir(parents=True, exist_ok=True)

print("Output folder:", output_dir.resolve())


Output folder: C:\Users\aiqiu\Documents\Nackademin\ML\wvs_mlproject\outputs\feature_selection


In [21]:
print("feature_selection exists:", "feature_selection" in globals())

if "feature_selection" in globals():
    print("Shape:", feature_selection.shape)

feature_selection exists: True
Shape: (372, 20)


In [22]:
print("Master dictionary:",
      (output_dir / "WVS_Master_Variable_Dictionary.csv").exists())

print("Feature selection:",
      (output_dir / "WVS_Feature_Selection.csv").exists())

Master dictionary: True
Feature selection: True


In [23]:
print("Current working directory:")
print(Path.cwd())

print("\nOutput folder:")
print(output_dir.resolve())

print("\nFiles inside output folder:")
for file in sorted(output_dir.iterdir()):
    print(file.name)

Current working directory:
c:\Users\aiqiu\Documents\Nackademin\ML\wvs_mlproject\notebooks

Output folder:
C:\Users\aiqiu\Documents\Nackademin\ML\wvs_mlproject\outputs\feature_selection

Files inside output folder:
WVS_Feature_Selection.csv
WVS_Master_Variable_Dictionary.csv


In [24]:
# Show all feature groups and their variable counts
group_summary = (
    feature_selection
    .groupby("Feature_group")
    .agg(
        Variable_count=("Variable", "count"),
        Variables=("Variable", lambda x: ", ".join(x))
    )
    .reset_index()
)

display(group_summary)

,Feature_group,Variable_count,Variables
0,Basic Demographic Background,15,"Q260, Q261, Q262, Q263, Q264, Q265, Q266, Q267..."
1,Child-rearing Values,11,"Q7, Q8, Q9, Q10, Q11, Q12, Q13, Q14, Q15, Q16,..."
2,Civic and Political Activism,4,"Q213, Q214, Q215, Q216"
3,Conventional Political Participation,4,"Q209, Q210, Q211, Q212"
4,Country-specific institutional confidence,17,"Q82_AFRICANUNION, Q82_APEC, Q82_ARABLEAGUE, Q8..."
5,Economic Security,7,"Q50, Q51, Q52, Q53, Q54, Q55, Q56"
6,Economic Values,6,"Q106, Q107, Q108, Q109, Q110, Q111"
7,Educational Background,12,"Q275, Q275A, Q275R, Q276, Q276A, Q276R, Q277, ..."
8,Electoral Integrity and Political Voice,12,"Q224, Q225, Q226, Q227, Q228, Q229, Q230, Q231..."
9,Employment and Occupational Background,6,"Q279, Q280, Q281, Q282, Q283, Q284"


In [25]:
personal_value_vars = ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6"]

feature_selection.loc[
    feature_selection["Variable"].isin(personal_value_vars),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(personal_value_vars),
    "Selection_reason"
] = (
    "Measures the importance of a major life domain; "
    "relevant for exploratory analysis of life satisfaction."
)

In [26]:
display(
    feature_selection[
        feature_selection["Variable"].isin(personal_value_vars)
    ][
        ["Variable", "Measurement_type",
         "Selection_decision", "Selection_reason"]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
0,Q1,Ordinal,Keep,Measures the importance of a major life domain...
1,Q2,Ordinal,Keep,Measures the importance of a major life domain...
2,Q3,Ordinal,Keep,Measures the importance of a major life domain...
3,Q4,Ordinal,Keep,Measures the importance of a major life domain...
4,Q5,Ordinal,Keep,Measures the importance of a major life domain...
5,Q6,Ordinal,Keep,Measures the importance of a major life domain...


In [27]:
child_values_vars = [
    f"Q{i}" for i in range(7, 18)
]

child_values = feature_selection[
    feature_selection["Variable"].isin(child_values_vars)
]

display(child_values)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
6,Q7,Child-rearing Values,Predictor candidate,Candidate,Social & Community,47,int64,0,0.0,5,397,-1: 122; -2: 172; -5: 103,1 | 2 | -2 | -1 | -5,Survey question / question-derived field,Binary,66.0,100.0,96823.0,Review,
7,Q8,Child-rearing Values,Predictor candidate,Candidate,Social & Community,48,int64,0,0.0,5,1038,-1: 464; -2: 481; -5: 93,1 | 2 | -2 | -1 | -5,Survey question / question-derived field,Binary,66.0,100.0,96182.0,Review,
8,Q9,Child-rearing Values,Predictor candidate,Candidate,Social & Community,49,int64,0,0.0,5,692,-1: 193; -2: 406; -5: 93,2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Binary,66.0,100.0,96528.0,Review,
9,Q10,Child-rearing Values,Predictor candidate,Candidate,Social & Community,50,int64,0,0.0,5,553,-1: 200; -2: 260; -5: 93,1 | 2 | -2 | -1 | -5,Survey question / question-derived field,Binary,66.0,100.0,96667.0,Review,
10,Q11,Child-rearing Values,Predictor candidate,Candidate,Social & Community,51,int64,0,0.0,5,1203,-1: 490; -2: 619; -5: 94,1 | 2 | -2 | -1 | -5,Survey question / question-derived field,Binary,66.0,100.0,96017.0,Review,
11,Q12,Child-rearing Values,Predictor candidate,Candidate,Social & Community,52,int64,0,0.0,5,643,-1: 219; -2: 331; -5: 93,2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Binary,66.0,100.0,96577.0,Review,
12,Q13,Child-rearing Values,Predictor candidate,Candidate,Social & Community,53,int64,0,0.0,5,1116,-1: 472; -2: 550; -5: 94,2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Binary,66.0,100.0,96104.0,Review,
13,Q14,Child-rearing Values,Predictor candidate,Candidate,Social & Community,54,int64,0,0.0,5,1131,-1: 429; -2: 608; -5: 94,2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Binary,66.0,100.0,96089.0,Review,
14,Q15,Child-rearing Values,Predictor candidate,Candidate,Social & Community,55,int64,0,0.0,5,1111,-1: 482; -2: 535; -5: 94,2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Binary,66.0,100.0,96109.0,Review,
15,Q16,Child-rearing Values,Predictor candidate,Candidate,Social & Community,56,int64,0,0.0,5,1124,-1: 428; -2: 602; -5: 94,2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Binary,66.0,100.0,96096.0,Review,


In [28]:
child_values_vars = [f"Q{i}" for i in range(7, 18)]

feature_selection.loc[
    feature_selection["Variable"].isin(child_values_vars),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(child_values_vars),
    "Selection_reason"
] = (
    "Measures whether the respondent selected a child-rearing "
    "quality among up to five choices; relevant for exploring "
    "family and personal value orientations."
)

In [29]:
display(
    feature_selection[
        feature_selection["Variable"].isin(child_values_vars)
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
6,Q7,Binary,Keep,Measures whether the respondent selected a chi...
7,Q8,Binary,Keep,Measures whether the respondent selected a chi...
8,Q9,Binary,Keep,Measures whether the respondent selected a chi...
9,Q10,Binary,Keep,Measures whether the respondent selected a chi...
10,Q11,Binary,Keep,Measures whether the respondent selected a chi...
11,Q12,Binary,Keep,Measures whether the respondent selected a chi...
12,Q13,Binary,Keep,Measures whether the respondent selected a chi...
13,Q14,Binary,Keep,Measures whether the respondent selected a chi...
14,Q15,Binary,Keep,Measures whether the respondent selected a chi...
15,Q16,Binary,Keep,Measures whether the respondent selected a chi...


In [30]:
social_tolerance_vars = [
    f"Q{i}" for i in range(18, 27)
]

social_tolerance = feature_selection[
    feature_selection["Variable"].isin(social_tolerance_vars)
]

display(social_tolerance)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
17,Q18,Social Tolerance,Predictor candidate,Candidate,Social & Community,58,int64,0,0.0,5,929,-1: 402; -2: 466; -5: 61,1 | 2 | -1 | -2 | -5,Survey question / question-derived field,Binary,66.0,100.00,96291.0,Review,
18,Q19,Social Tolerance,Predictor candidate,Candidate,Social & Community,59,int64,0,0.0,5,1156,-1: 518; -2: 481; -5: 157,2 | 1 | -1 | -2 | -5,Survey question / question-derived field,Binary,66.0,100.00,96064.0,Review,
19,Q20,Social Tolerance,Predictor candidate,Candidate,Social & Community,60,int64,0,0.0,5,1461,-1: 774; -2: 557; -5: 130,1 | 2 | -1 | -2 | -5,Survey question / question-derived field,Binary,66.0,100.00,95759.0,Review,
20,Q21,Social Tolerance,Predictor candidate,Candidate,Social & Community,61,int64,0,0.0,5,1306,-1: 668; -2: 493; -5: 145,2 | 1 | -1 | -2 | -5,Survey question / question-derived field,Binary,66.0,100.00,95914.0,Review,
21,Q22,Social Tolerance,Predictor candidate,Candidate,Social & Community,62,int64,0,0.0,6,4137,-1: 808; -2: 546; -4: 2699; -5: 84,2 | 1 | -1 | -2 | -5 | -4,Survey question / question-derived field,Binary,64.0,96.97,93083.0,Review,
22,Q23,Social Tolerance,Predictor candidate,Candidate,Social & Community,63,int64,0,0.0,5,1193,-1: 551; -2: 496; -5: 146,2 | 1 | -1 | -2 | -5,Survey question / question-derived field,Binary,66.0,100.00,96027.0,Review,
23,Q24,Social Tolerance,Predictor candidate,Candidate,Social & Community,64,int64,0,0.0,5,1007,-1: 465; -2: 478; -5: 64,1 | 2 | -1 | -2 | -5,Survey question / question-derived field,Binary,66.0,100.00,96213.0,Review,
24,Q25,Social Tolerance,Predictor candidate,Candidate,Social & Community,65,int64,0,0.0,6,2443,-1: 611; -2: 538; -4: 1200; -5: 94,2 | 1 | -1 | -2 | -4 | -5,Survey question / question-derived field,Binary,65.0,98.48,94777.0,Review,
25,Q26,Social Tolerance,Predictor candidate,Candidate,Social & Community,66,int64,0,0.0,5,1190,-1: 560; -2: 534; -5: 96,2 | 1 | -1 | -2 | -5,Survey question / question-derived field,Binary,66.0,100.00,96030.0,Review,


In [31]:
social_tolerance_keep = [
    "Q18", "Q19", "Q20", "Q21",
    "Q23", "Q24", "Q26"
]

social_tolerance_optional = ["Q22", "Q25"]

feature_selection.loc[
    feature_selection["Variable"].isin(social_tolerance_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(social_tolerance_keep),
    "Selection_reason"
] = (
    "Measures whether the respondent mentioned a social group "
    "as an undesirable neighbour; relevant for exploring "
    "social attitudes and life satisfaction."
)

feature_selection.loc[
    feature_selection["Variable"].isin(social_tolerance_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(social_tolerance_optional),
    "Selection_reason"
] = (
    "Potentially relevant social attitude predictor, but has "
    "country-specific Not asked responses and lower country coverage; "
    "review missingness before modeling."
)

In [32]:
display(
    feature_selection[
        feature_selection["Variable"].isin(
            [f"Q{i}" for i in range(18, 27)]
        )
    ][
        [
            "Variable",
            "Measurement_type",
            "Countries_with_valid_responses",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Countries_with_valid_responses,Selection_decision,Selection_reason
17,Q18,Binary,66.0,Keep,Measures whether the respondent mentioned a so...
18,Q19,Binary,66.0,Keep,Measures whether the respondent mentioned a so...
19,Q20,Binary,66.0,Keep,Measures whether the respondent mentioned a so...
20,Q21,Binary,66.0,Keep,Measures whether the respondent mentioned a so...
21,Q22,Binary,64.0,Optional,Potentially relevant social attitude predictor...
22,Q23,Binary,66.0,Keep,Measures whether the respondent mentioned a so...
23,Q24,Binary,66.0,Keep,Measures whether the respondent mentioned a so...
24,Q25,Binary,65.0,Optional,Potentially relevant social attitude predictor...
25,Q26,Binary,66.0,Keep,Measures whether the respondent mentioned a so...


In [33]:
family_gender_vars = [
    f"Q{i}" for i in range(27, 39)
]

family_gender = feature_selection[
    feature_selection["Variable"].isin(family_gender_vars)
]

display(family_gender)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
26,Q27,Family and Gender Norms,Predictor candidate,Candidate,Social & Community,67,int64,0,0.0,7,1676,-1: 1156; -2: 359; -5: 161,3 | 1 | 2 | 4 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.00,95544.0,Review,
27,Q28,Family and Gender Norms,Predictor candidate,Candidate,Social & Community,68,int64,0,0.0,7,1871,-1: 1337; -2: 373; -5: 161,3 | 1 | 2 | 4 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.00,95349.0,Review,
28,Q29,Family and Gender Norms,Predictor candidate,Candidate,Social & Community,69,int64,0,0.0,7,2641,-1: 2107; -2: 384; -5: 150,3 | 4 | 2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.00,94579.0,Review,
29,Q30,Family and Gender Norms,Predictor candidate,Candidate,Social & Community,70,int64,0,0.0,7,1428,-1: 1033; -2: 256; -5: 139,3 | 4 | 1 | 2 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.00,95792.0,Review,
30,Q31,Family and Gender Norms,Predictor candidate,Candidate,Social & Community,71,int64,0,0.0,7,2068,-1: 1625; -2: 309; -5: 134,3 | 4 | 2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.00,95152.0,Review,
31,Q32,Family and Gender Norms,Predictor candidate,Candidate,Social & Community,72,int64,0,0.0,7,3201,-1: 2563; -2: 459; -5: 179,3 | -2 | 4 | 2 | 1 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.00,94019.0,Review,
32,Q33,Family and Gender Norms,Predictor candidate,Candidate,Social & Community,73,int64,0,0.0,8,736,-1: 503; -2: 122; -5: 111,4 | 5 | 3 | 1 | -2 | 2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.00,96484.0,Review,
33,Q34,Family and Gender Norms,Predictor candidate,Candidate,Social & Community,75,int64,0,0.0,8,913,-1: 626; -2: 177; -5: 110,2 | 4 | 5 | 1 | 3 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.00,96307.0,Review,
34,Q35,Family and Gender Norms,Predictor candidate,Candidate,Social & Community,77,int64,0,0.0,8,1464,-1: 1140; -2: 199; -5: 125,3 | 4 | 2 | 5 | 1 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.00,95756.0,Review,
35,Q36,Family and Gender Norms,Predictor candidate,Candidate,Social & Community,79,int64,0,0.0,9,16459,-1: 4646; -2: 508; -4: 11160; -5: 145,3 | 2 | 4 | 1 | -2 | 5 | -1 | -5,Survey question / question-derived field,Ordinal,57.0,86.36,80761.0,Review,


In [34]:
family_gender_keep = [
    f"Q{i}" for i in range(27, 36)
] + ["Q37", "Q38"]

family_gender_optional = ["Q36"]

feature_selection.loc[
    feature_selection["Variable"].isin(family_gender_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(family_gender_keep),
    "Selection_reason"
] = (
    "Measures family, gender-role or intergenerational attitudes; "
    "relevant for exploring social values associated with "
    "life satisfaction."
)

feature_selection.loc[
    feature_selection["Variable"].isin(family_gender_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(family_gender_optional),
    "Selection_reason"
] = (
    "Relevant attitude predictor, but substantial country-specific "
    "Not asked responses and lower country coverage; review "
    "cross-country comparability before modeling."
)

In [35]:
display(
    feature_selection[
        feature_selection["Variable"].isin(
            [f"Q{i}" for i in range(27, 39)]
        )
    ][
        [
            "Variable",
            "Measurement_type",
            "Countries_with_valid_responses",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Countries_with_valid_responses,Selection_decision,Selection_reason
26,Q27,Ordinal,66.0,Keep,"Measures family, gender-role or intergeneratio..."
27,Q28,Ordinal,66.0,Keep,"Measures family, gender-role or intergeneratio..."
28,Q29,Ordinal,66.0,Keep,"Measures family, gender-role or intergeneratio..."
29,Q30,Ordinal,66.0,Keep,"Measures family, gender-role or intergeneratio..."
30,Q31,Ordinal,66.0,Keep,"Measures family, gender-role or intergeneratio..."
31,Q32,Ordinal,66.0,Keep,"Measures family, gender-role or intergeneratio..."
32,Q33,Ordinal,66.0,Keep,"Measures family, gender-role or intergeneratio..."
33,Q34,Ordinal,66.0,Keep,"Measures family, gender-role or intergeneratio..."
34,Q35,Ordinal,66.0,Keep,"Measures family, gender-role or intergeneratio..."
35,Q36,Ordinal,57.0,Optional,"Relevant attitude predictor, but substantial c..."


In [36]:
work_attitude_vars = [
    f"Q{i}" for i in range(39, 46)
]

work_attitudes = feature_selection[
    feature_selection["Variable"].isin(work_attitude_vars)
]

display(work_attitudes)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
38,Q39,Work and Social Attitudes,Predictor candidate,Candidate,Personal Values & Wellbeing,82,int64,0,0.0,8,948,-1: 615; -2: 212; -5: 121,2 | 3 | 1 | -2 | 4 | 5 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.0,96272.0,Review,
39,Q40,Work and Social Attitudes,Predictor candidate,Candidate,Personal Values & Wellbeing,83,int64,0,0.0,8,852,-1: 545; -2: 181; -5: 126,3 | 2 | 4 | 1 | 5 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.0,96368.0,Review,
40,Q41,Work and Social Attitudes,Predictor candidate,Candidate,Personal Values & Wellbeing,84,int64,0,0.0,8,790,-1: 469; -2: 200; -5: 121,3 | 2 | 4 | 1 | 5 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.0,96430.0,Review,
41,Q42,Work and Social Attitudes,Predictor candidate,Candidate,Personal Values & Wellbeing,85,int64,0,0.0,6,3318,-1: 2256; -2: 769; -5: 293,2 | 1 | 3 | -2 | -1 | -5,Survey question / question-derived field,Nominal,66.0,100.0,93902.0,Review,
42,Q43,Work and Social Attitudes,Predictor candidate,Candidate,Personal Values & Wellbeing,86,int64,0,0.0,6,2137,-1: 1557; -2: 369; -5: 211,2 | 1 | 3 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.0,95083.0,Review,
43,Q44,Work and Social Attitudes,Predictor candidate,Candidate,Personal Values & Wellbeing,87,int64,0,0.0,6,2063,-1: 1541; -2: 333; -5: 189,3 | 1 | 2 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.0,95157.0,Review,
44,Q45,Work and Social Attitudes,Predictor candidate,Candidate,Personal Values & Wellbeing,88,int64,0,0.0,6,2724,-1: 1972; -2: 522; -5: 230,1 | 2 | 3 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,66.0,100.0,94496.0,Review,


In [37]:
work_attitude_vars = [
    f"Q{i}" for i in range(39, 46)
]

# Correct Q42 measurement type
master_dictionary.loc[
    master_dictionary["Variable"] == "Q42",
    "Measurement_type"
] = "Nominal"

feature_selection.loc[
    feature_selection["Variable"] == "Q42",
    "Measurement_type"
] = "Nominal"

# Keep all seven variables
feature_selection.loc[
    feature_selection["Variable"].isin(work_attitude_vars),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(work_attitude_vars),
    "Selection_reason"
] = (
    "Measures work-related values, social attitudes or "
    "preferences concerning possible societal changes; "
    "relevant for exploratory analysis of life satisfaction."
)

In [38]:
display(
    feature_selection[
        feature_selection["Variable"].isin(work_attitude_vars)
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
38,Q39,Ordinal,Keep,"Measures work-related values, social attitudes..."
39,Q40,Ordinal,Keep,"Measures work-related values, social attitudes..."
40,Q41,Ordinal,Keep,"Measures work-related values, social attitudes..."
41,Q42,Nominal,Keep,"Measures work-related values, social attitudes..."
42,Q43,Ordinal,Keep,"Measures work-related values, social attitudes..."
43,Q44,Ordinal,Keep,"Measures work-related values, social attitudes..."
44,Q45,Ordinal,Keep,"Measures work-related values, social attitudes..."


In [39]:
wellbeing_vars = ["Q46", "Q47", "Q48"]

wellbeing = feature_selection[
    feature_selection["Variable"].isin(wellbeing_vars)
]

display(wellbeing)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
45,Q46,Wellbeing,Predictor candidate,Candidate,Personal Values & Wellbeing,89,int64,0,0.0,7,651,-1: 309; -2: 222; -5: 120,1 | 2 | 3 | 4 | -1 | -2 | -5,Survey question / question-derived field,Ordinal,66.0,100.0,96569.0,Review,
46,Q47,Wellbeing,Predictor candidate,Candidate,Personal Values & Wellbeing,90,int64,0,0.0,8,267,-1: 80; -2: 93; -5: 94,3 | 1 | 2 | 4 | 5 | -1 | -2 | -5,Survey question / question-derived field,Ordinal,66.0,100.0,96953.0,Review,
47,Q48,Wellbeing,Predictor candidate,Candidate,Personal Values & Wellbeing,91,int64,0,0.0,13,917,-1: 575; -2: 236; -5: 106,10 | 9 | 8 | 7 | 3 | 5 | 6 | 1,Survey question / question-derived field,Ordinal,66.0,100.0,96303.0,Review,


In [40]:
wellbeing_keep = ["Q47", "Q48"]
wellbeing_optional = ["Q46"]

feature_selection.loc[
    feature_selection["Variable"].isin(wellbeing_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(wellbeing_keep),
    "Selection_reason"
] = (
    "Measures subjective health or perceived freedom and control; "
    "relevant for exploring factors associated with life satisfaction."
)

feature_selection.loc[
    feature_selection["Variable"].isin(wellbeing_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(wellbeing_optional),
    "Selection_reason"
] = (
    "Measures subjective happiness, which is conceptually close "
    "to life satisfaction; retain for exploratory analysis and "
    "compare models with and without this predictor."
)

In [41]:
display(
    feature_selection[
        feature_selection["Variable"].isin(
            ["Q46", "Q47", "Q48"]
        )
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
45,Q46,Ordinal,Optional,"Measures subjective happiness, which is concep..."
46,Q47,Ordinal,Keep,Measures subjective health or perceived freedo...
47,Q48,Ordinal,Keep,Measures subjective health or perceived freedo...


In [42]:
trust_vars = [
    f"Q{i}" for i in range(57, 64)
]

trust_features = feature_selection[
    feature_selection["Variable"].isin(trust_vars)
]

display(trust_features)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
55,Q57,General and interpersonal trust,Predictor candidate,Candidate,Social & Community,100,int64,0,0.0,5,1337,-1: 737; -2: 310; -5: 290,2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Binary,NaN,NaN,NaN,Review,
56,Q58,General and interpersonal trust,Predictor candidate,Candidate,Social & Community,101,int64,0,0.0,7,332,-1: 102; -2: 125; -5: 105,1 | 2 | 3 | 4 | -1 | -2 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
57,Q59,General and interpersonal trust,Predictor candidate,Candidate,Social & Community,102,int64,0,0.0,7,827,-1: 534; -2: 184; -5: 109,2 | 3 | 4 | 1 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
58,Q60,General and interpersonal trust,Predictor candidate,Candidate,Social & Community,103,int64,0,0.0,7,585,-1: 284; -2: 195; -5: 106,2 | 3 | 1 | 4 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
59,Q61,General and interpersonal trust,Predictor candidate,Candidate,Social & Community,104,int64,0,0.0,7,1361,-1: 1009; -2: 223; -5: 129,2 | 3 | 4 | 1 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
60,Q62,General and interpersonal trust,Predictor candidate,Candidate,Social & Community,105,int64,0,0.0,7,4076,-1: 3049; -2: 705; -5: 322,2 | 3 | 1 | -2 | 4 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
61,Q63,General and interpersonal trust,Predictor candidate,Candidate,Social & Community,106,int64,0,0.0,7,4078,-1: 3182; -2: 632; -5: 264,2 | 3 | 1 | 4 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,


In [43]:
trust_vars = [f"Q{i}" for i in range(57, 64)]

feature_selection.loc[
    feature_selection["Variable"].isin(trust_vars),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(trust_vars),
    "Selection_reason"
] = (
    "Measures generalised or group-specific interpersonal trust; "
    "relevant for exploring social trust and life satisfaction."
)

In [44]:
display(
    feature_selection[
        feature_selection["Variable"].isin(trust_vars)
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
55,Q57,Binary,Keep,Measures generalised or group-specific interpe...
56,Q58,Ordinal,Keep,Measures generalised or group-specific interpe...
57,Q59,Ordinal,Keep,Measures generalised or group-specific interpe...
58,Q60,Ordinal,Keep,Measures generalised or group-specific interpe...
59,Q61,Ordinal,Keep,Measures generalised or group-specific interpe...
60,Q62,Ordinal,Keep,Measures generalised or group-specific interpe...
61,Q63,Ordinal,Keep,Measures generalised or group-specific interpe...


In [45]:
institutional_vars = [
    f"Q{i}" for i in range(64, 83)
]

institutional_features = feature_selection[
    feature_selection["Variable"].isin(institutional_vars)
]

display(institutional_features)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
62,Q64,Institutional confidence,Predictor candidate,Candidate,Political & Institutional,107,int64,0,0.0,7,2036,-1: 1458; -2: 314; -5: 264,1 | 4 | 2 | 3 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
63,Q65,Institutional confidence,Predictor candidate,Candidate,Political & Institutional,108,int64,0,0.0,8,4323,-1: 1648; -2: 276; -4: 2204; -5: 195,-4 | 4 | 1 | 3 | 2 | -1 | -2 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
64,Q66,Institutional confidence,Predictor candidate,Candidate,Political & Institutional,109,int64,0,0.0,7,2210,-1: 1781; -2: 262; -5: 167,1 | 3 | 4 | 2 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
65,Q67,Institutional confidence,Predictor candidate,Candidate,Political & Institutional,110,int64,0,0.0,7,1448,-1: 1027; -2: 243; -5: 178,1 | 3 | 4 | 2 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
66,Q68,Institutional confidence,Predictor candidate,Candidate,Political & Institutional,111,int64,0,0.0,7,7255,-1: 6322; -2: 621; -5: 312,1 | 4 | 3 | 2 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
67,Q69,Institutional confidence,Predictor candidate,Candidate,Political & Institutional,112,int64,0,0.0,8,2412,-1: 813; -2: 249; -4: 1200; -5: 150,1 | 3 | 2 | 4 | -2 | -1 | -4 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
68,Q70,Institutional confidence,Predictor candidate,Candidate,Political & Institutional,113,int64,0,0.0,8,3511,-1: 1828; -2: 316; -4: 1200; -5: 167,1 | 3 | 2 | 4 | -2 | -1 | -4 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
69,Q71,Institutional confidence,Predictor candidate,Candidate,Political & Institutional,114,int64,0,0.0,8,3137,-1: 1437; -2: 341; -4: 1200; -5: 159,1 | 4 | 3 | 2 | -2 | -1 | -4 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
70,Q72,Institutional confidence,Predictor candidate,Candidate,Political & Institutional,115,int64,0,0.0,7,3305,-1: 2653; -2: 466; -5: 186,1 | 4 | 3 | 2 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
71,Q73,Institutional confidence,Predictor candidate,Candidate,Political & Institutional,116,int64,0,0.0,7,3479,-1: 2819; -2: 475; -5: 185,1 | 4 | 3 | 2 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,


In [46]:
institutional_keep = [
    f"Q{i}" for i in range(64, 82)
]

institutional_optional = ["Q82"]

feature_selection.loc[
    feature_selection["Variable"].isin(institutional_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(institutional_keep),
    "Selection_reason"
] = (
    "Measures confidence in social, political, economic or "
    "other institutions; relevant for exploring institutional "
    "trust and life satisfaction."
)

feature_selection.loc[
    feature_selection["Variable"].isin(institutional_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(institutional_optional),
    "Selection_reason"
] = (
    "Measures confidence in a country-specific international "
    "or regional organization; review cross-country comparability "
    "and missingness before modeling."
)

In [47]:
institutional_vars = [
    f"Q{i}" for i in range(64, 83)
]

display(
    feature_selection[
        feature_selection["Variable"].isin(institutional_vars)
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
62,Q64,Ordinal,Keep,"Measures confidence in social, political, econ..."
63,Q65,Ordinal,Keep,"Measures confidence in social, political, econ..."
64,Q66,Ordinal,Keep,"Measures confidence in social, political, econ..."
65,Q67,Ordinal,Keep,"Measures confidence in social, political, econ..."
66,Q68,Ordinal,Keep,"Measures confidence in social, political, econ..."
67,Q69,Ordinal,Keep,"Measures confidence in social, political, econ..."
68,Q70,Ordinal,Keep,"Measures confidence in social, political, econ..."
69,Q71,Ordinal,Keep,"Measures confidence in social, political, econ..."
70,Q72,Ordinal,Keep,"Measures confidence in social, political, econ..."
71,Q73,Ordinal,Keep,"Measures confidence in social, political, econ..."


In [48]:
membership_vars = [
    f"Q{i}" for i in range(94, 106)
]

membership_features = feature_selection[
    feature_selection["Variable"].isin(membership_vars)
]

display(membership_features)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
108,Q94,Organizational membership,Predictor candidate,Candidate,Social & Community,154,int64,0,0.0,6,1094,-1: 583; -2: 320; -5: 191,0 | 1 | 2 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
109,Q95,Organizational membership,Predictor candidate,Candidate,Social & Community,156,int64,0,0.0,6,1138,-1: 645; -2: 260; -5: 233,0 | 2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
110,Q96,Organizational membership,Predictor candidate,Candidate,Social & Community,158,int64,0,0.0,6,1214,-1: 719; -2: 258; -5: 237,0 | 2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
111,Q97,Organizational membership,Predictor candidate,Candidate,Social & Community,160,int64,0,0.0,6,1391,-1: 845; -2: 290; -5: 256,0 | 2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
112,Q98,Organizational membership,Predictor candidate,Candidate,Social & Community,162,int64,0,0.0,6,1344,-1: 800; -2: 314; -5: 230,0 | 2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
113,Q99,Organizational membership,Predictor candidate,Candidate,Social & Community,164,int64,0,0.0,6,1426,-1: 898; -2: 322; -5: 206,0 | 1 | 2 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
114,Q100,Organizational membership,Predictor candidate,Candidate,Social & Community,166,int64,0,0.0,6,1597,-1: 977; -2: 347; -5: 273,0 | 2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
115,Q101,Organizational membership,Predictor candidate,Candidate,Social & Community,168,int64,0,0.0,6,1403,-1: 847; -2: 325; -5: 231,0 | 2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
116,Q102,Organizational membership,Predictor candidate,Candidate,Social & Community,170,int64,0,0.0,6,1715,-1: 1077; -2: 369; -5: 269,0 | 1 | 2 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
117,Q103,Organizational membership,Predictor candidate,Candidate,Social & Community,172,int64,0,0.0,6,1602,-1: 995; -2: 350; -5: 257,0 | 2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,


In [49]:
membership_keep = [
    f"Q{i}" for i in range(94, 104)
]

membership_optional = ["Q104", "Q105"]

feature_selection.loc[
    feature_selection["Variable"].isin(membership_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(membership_keep),
    "Selection_reason"
] = (
    "Measures participation and membership intensity in different "
    "social, political, religious or voluntary organizations."
)

feature_selection.loc[
    feature_selection["Variable"].isin(membership_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(membership_optional),
    "Selection_reason"
] = (
    "Potentially relevant organizational participation predictor, "
    "but requires additional missingness and data-quality review."
)

In [50]:
membership_vars = [
    f"Q{i}" for i in range(94, 106)
]

display(
    feature_selection[
        feature_selection["Variable"].isin(membership_vars)
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
108,Q94,Ordinal,Keep,Measures participation and membership intensit...
109,Q95,Ordinal,Keep,Measures participation and membership intensit...
110,Q96,Ordinal,Keep,Measures participation and membership intensit...
111,Q97,Ordinal,Keep,Measures participation and membership intensit...
112,Q98,Ordinal,Keep,Measures participation and membership intensit...
113,Q99,Ordinal,Keep,Measures participation and membership intensit...
114,Q100,Ordinal,Keep,Measures participation and membership intensit...
115,Q101,Ordinal,Keep,Measures participation and membership intensit...
116,Q102,Ordinal,Keep,Measures participation and membership intensit...
117,Q103,Ordinal,Keep,Measures participation and membership intensit...


In [51]:
corruption_vars = [
    f"Q{i}" for i in range(112, 121)
]

corruption_features = feature_selection[
    feature_selection["Variable"].isin(corruption_vars)
]

display(corruption_features)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
138,Q112,Perceived Corruption,Predictor candidate,Candidate,Political & Institutional,184,int64,0,0.0,13,1645,-1: 1168; -2: 326; -5: 151,2 | 10 | 7 | 5 | 6 | 8 | 1 | 9,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
139,Q113,Perceived Corruption,Predictor candidate,Candidate,Political & Institutional,185,int64,0,0.0,8,6589,-1: 3294; -2: 716; -4: 2400; -5: 179,-2 | 3 | 2 | 1 | 4 | -1 | -5 | -4,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
140,Q114,Perceived Corruption,Predictor candidate,Candidate,Political & Institutional,186,int64,0,0.0,8,7426,-1: 4144; -2: 706; -4: 2400; -5: 176,-2 | 3 | 2 | 1 | 4 | -1 | -4 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
141,Q115,Perceived Corruption,Predictor candidate,Candidate,Political & Institutional,187,int64,0,0.0,8,9071,-1: 3762; -2: 701; -4: 4412; -5: 196,-2 | 3 | 2 | 1 | 4 | -1 | -5 | -4,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
142,Q116,Perceived Corruption,Predictor candidate,Candidate,Political & Institutional,188,int64,0,0.0,8,6521,-1: 3251; -2: 691; -4: 2400; -5: 179,-2 | 3 | 2 | 1 | 4 | -1 | -5 | -4,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
143,Q117,Perceived Corruption,Predictor candidate,Candidate,Political & Institutional,189,int64,0,0.0,8,8985,-1: 5560; -2: 834; -4: 2400; -5: 191,-2 | 3 | 2 | 1 | 4 | -1 | -5 | -4,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
144,Q118,Perceived Corruption,Predictor candidate,Candidate,Political & Institutional,190,int64,0,0.0,8,5392,-1: 3215; -2: 749; -4: 1200; -5: 228,1 | 3 | 2 | -2 | 4 | -1 | -5 | -4,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
145,Q119,Perceived Corruption,Predictor candidate,Candidate,Political & Institutional,191,int64,0,0.0,9,5812,-1: 2162; -2: 425; -4: 3056; -5: 169,2 | 3 | 4 | 0 | 1 | -2 | -1 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,
146,Q120,Perceived Corruption,Predictor candidate,Candidate,Political & Institutional,192,int64,0,0.0,14,4137,-1: 2271; -2: 487; -4: 1200; -5: 179,6 | 2 | 7 | 5 | 10 | 4 | 8 | 3,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,


In [52]:
corruption_vars = [
    f"Q{i}" for i in range(112, 121)
]

feature_selection.loc[
    feature_selection["Variable"].isin(corruption_vars),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(corruption_vars),
    "Selection_reason"
] = (
    "Measures perceived corruption, beliefs about corruption "
    "in different groups, bribery experiences or perceived "
    "accountability risk."
)

# Q119 is nominal because "Hard to say" is not an ordered level.
feature_selection.loc[
    feature_selection["Variable"] == "Q119",
    "Measurement_type"
] = "Nominal"

master_dictionary.loc[
    master_dictionary["Variable"] == "Q119",
    "Measurement_type"
] = "Nominal"

In [53]:
display(
    feature_selection[
        feature_selection["Variable"].isin(corruption_vars)
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
138,Q112,Ordinal,Keep,"Measures perceived corruption, beliefs about c..."
139,Q113,Ordinal,Keep,"Measures perceived corruption, beliefs about c..."
140,Q114,Ordinal,Keep,"Measures perceived corruption, beliefs about c..."
141,Q115,Ordinal,Keep,"Measures perceived corruption, beliefs about c..."
142,Q116,Ordinal,Keep,"Measures perceived corruption, beliefs about c..."
143,Q117,Ordinal,Keep,"Measures perceived corruption, beliefs about c..."
144,Q118,Ordinal,Keep,"Measures perceived corruption, beliefs about c..."
145,Q119,Nominal,Keep,"Measures perceived corruption, beliefs about c..."
146,Q120,Ordinal,Keep,"Measures perceived corruption, beliefs about c..."


In [54]:
immigration_vars = [
    f"Q{i}" for i in range(121, 131)
]

immigration_features = feature_selection[
    feature_selection["Variable"].isin(immigration_vars)
]

display(immigration_features)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
147,Q121,Immigration Attitudes,Predictor candidate,Candidate,Social & Community,193,int64,0,0.0,8,2533,-1: 1947; -2: 439; -5: 147,5 | 4 | 3 | 2 | -2 | 1 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
148,Q122,Immigration Attitudes,Predictor candidate,Candidate,Social & Community,194,int64,0,0.0,6,2709,-1: 2252; -2: 285; -5: 172,1 | 2 | 0 | -2 | -1 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,
149,Q123,Immigration Attitudes,Predictor candidate,Candidate,Social & Community,195,int64,0,0.0,6,3141,-1: 2603; -2: 374; -5: 164,-2 | 2 | 1 | 0 | -1 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,
150,Q124,Immigration Attitudes,Predictor candidate,Candidate,Social & Community,196,int64,0,0.0,6,2727,-1: 2255; -2: 312; -5: 160,0 | 1 | 2 | -2 | -1 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,
151,Q125,Immigration Attitudes,Predictor candidate,Candidate,Social & Community,197,int64,0,0.0,7,7516,-1: 3799; -2: 511; -4: 3036; -5: 170,2 | 0 | 1 | -2 | -1 | -4 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,
152,Q126,Immigration Attitudes,Predictor candidate,Candidate,Social & Community,198,int64,0,0.0,6,3079,-1: 2530; -2: 374; -5: 175,0 | 1 | 2 | -2 | -1 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,
153,Q127,Immigration Attitudes,Predictor candidate,Candidate,Social & Community,199,int64,0,0.0,6,2800,-1: 2238; -2: 394; -5: 168,-2 | 2 | 1 | 0 | -1 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,
154,Q128,Immigration Attitudes,Predictor candidate,Candidate,Social & Community,200,int64,0,0.0,6,2305,-1: 1844; -2: 310; -5: 151,2 | 0 | 1 | -1 | -2 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,
155,Q129,Immigration Attitudes,Predictor candidate,Candidate,Social & Community,201,int64,0,0.0,6,2824,-1: 2334; -2: 335; -5: 155,0 | 1 | 2 | -2 | -1 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,
156,Q130,Immigration Attitudes,Predictor candidate,Candidate,Social & Community,202,int64,0,0.0,8,5508,-1: 1786; -2: 507; -4: 3036; -5: 179,2 | 3 | 1 | -2 | 4 | -1 | -4 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,


In [55]:
immigration_vars = [
    f"Q{i}" for i in range(121, 131)
]

# Q122-Q129 are nominal because "Hard to say"
# is not an ordered response level.
immigration_nominal = [
    f"Q{i}" for i in range(122, 130)
]

master_dictionary.loc[
    master_dictionary["Variable"].isin(immigration_nominal),
    "Measurement_type"
] = "Nominal"

feature_selection.loc[
    feature_selection["Variable"].isin(immigration_nominal),
    "Measurement_type"
] = "Nominal"

# Keep the main candidates
immigration_keep = [
    "Q121", "Q122", "Q123", "Q124",
    "Q126", "Q127", "Q128", "Q129"
]

immigration_optional = ["Q125", "Q130"]

feature_selection.loc[
    feature_selection["Variable"].isin(immigration_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(immigration_keep),
    "Selection_reason"
] = (
    "Measures perceptions of immigration and its economic, "
    "cultural or social effects; relevant for exploring "
    "immigration attitudes and life satisfaction."
)

feature_selection.loc[
    feature_selection["Variable"].isin(immigration_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(immigration_optional),
    "Selection_reason"
] = (
    "Relevant immigration attitude predictor, but has substantial "
    "country-specific Not asked responses; review coverage "
    "before modeling."
)

In [56]:
display(
    feature_selection[
        feature_selection["Variable"].isin(immigration_vars)
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
147,Q121,Ordinal,Keep,Measures perceptions of immigration and its ec...
148,Q122,Nominal,Keep,Measures perceptions of immigration and its ec...
149,Q123,Nominal,Keep,Measures perceptions of immigration and its ec...
150,Q124,Nominal,Keep,Measures perceptions of immigration and its ec...
151,Q125,Nominal,Optional,"Relevant immigration attitude predictor, but h..."
152,Q126,Nominal,Keep,Measures perceptions of immigration and its ec...
153,Q127,Nominal,Keep,Measures perceptions of immigration and its ec...
154,Q128,Nominal,Keep,Measures perceptions of immigration and its ec...
155,Q129,Nominal,Keep,Measures perceptions of immigration and its ec...
156,Q130,Ordinal,Optional,"Relevant immigration attitude predictor, but h..."


In [57]:
security_vars = [
    f"Q{i}" for i in range(131, 142)
]

security_features = feature_selection[
    feature_selection["Variable"].isin(security_vars)
]

display(security_features)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
157,Q131,Personal and Neighborhood Security,Predictor candidate,Candidate,Security,203,int64,0,0.0,7,670,-1: 367; -2: 167; -5: 136,1 | 2 | 3 | 4 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
158,Q132,Personal and Neighborhood Security,Predictor candidate,Candidate,Security,204,int64,0,0.0,8,4500,-1: 1138; -2: 172; -4: 3036; -5: 154,4 | 3 | 2 | 1 | -1 | -2 | -4 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
159,Q133,Personal and Neighborhood Security,Predictor candidate,Candidate,Security,205,int64,0,0.0,8,4485,-1: 1106; -2: 203; -4: 3036; -5: 140,4 | 2 | 1 | 3 | -2 | -1 | -4 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
160,Q134,Personal and Neighborhood Security,Predictor candidate,Candidate,Security,206,int64,0,0.0,8,6831,-1: 2116; -2: 326; -4: 4236; -5: 153,4 | 3 | -2 | 2 | 1 | -1 | -4 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
161,Q135,Personal and Neighborhood Security,Predictor candidate,Candidate,Security,207,int64,0,0.0,8,6040,-1: 2464; -2: 359; -4: 3036; -5: 181,4 | 3 | 2 | -2 | 1 | -1 | -4 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
162,Q136,Personal and Neighborhood Security,Predictor candidate,Candidate,Security,208,int64,0,0.0,8,7609,-1: 3964; -2: 439; -4: 3036; -5: 170,4 | 3 | 2 | 1 | -2 | -1 | -4 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
163,Q137,Personal and Neighborhood Security,Predictor candidate,Candidate,Security,209,int64,0,0.0,8,4986,-1: 1509; -2: 283; -4: 3036; -5: 158,4 | 3 | 1 | 2 | -2 | -1 | -4 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
164,Q138,Personal and Neighborhood Security,Predictor candidate,Candidate,Security,210,int64,0,0.0,8,8694,-1: 3866; -2: 422; -4: 4236; -5: 170,4 | -2 | 3 | 2 | -1 | 1 | -4 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
165,Q139,Personal and Neighborhood Security,Predictor candidate,Candidate,Security,211,int64,0,0.0,6,8266,-1: 466; -2: 494; -4: 6092; -5: 1214,2 | 1 | -2 | -1 | -4 | -5,Survey question / question-derived field,Binary,NaN,NaN,NaN,Review,
166,Q140,Personal and Neighborhood Security,Predictor candidate,Candidate,Security,212,int64,0,0.0,6,8106,-1: 429; -2: 481; -4: 6092; -5: 1104,2 | 1 | -2 | -1 | -4 | -5,Survey question / question-derived field,Binary,NaN,NaN,NaN,Review,


In [58]:
security_vars = [
    f"Q{i}" for i in range(131, 142)
]

security_keep = [
    f"Q{i}" for i in range(131, 139)
]

security_optional = [
    "Q139", "Q140", "Q141"
]

feature_selection.loc[
    feature_selection["Variable"].isin(security_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(security_keep),
    "Selection_reason"
] = (
    "Measures personal safety or the frequency of security-related "
    "problems in the respondent's neighborhood."
)

feature_selection.loc[
    feature_selection["Variable"].isin(security_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(security_optional),
    "Selection_reason"
] = (
    "Measures security-related behavioural adaptations; "
    "requires additional country coverage and missingness review."
)

In [59]:
display(
    feature_selection[
        feature_selection["Variable"].isin(security_vars)
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
157,Q131,Ordinal,Keep,Measures personal safety or the frequency of s...
158,Q132,Ordinal,Keep,Measures personal safety or the frequency of s...
159,Q133,Ordinal,Keep,Measures personal safety or the frequency of s...
160,Q134,Ordinal,Keep,Measures personal safety or the frequency of s...
161,Q135,Ordinal,Keep,Measures personal safety or the frequency of s...
162,Q136,Ordinal,Keep,Measures personal safety or the frequency of s...
163,Q137,Ordinal,Keep,Measures personal safety or the frequency of s...
164,Q138,Ordinal,Keep,Measures personal safety or the frequency of s...
165,Q139,Binary,Optional,Measures security-related behavioural adaptati...
166,Q140,Binary,Optional,Measures security-related behavioural adaptati...


In [60]:
worry_vars = [
    f"Q{i}" for i in range(142, 149)
]

worry_features = feature_selection[
    feature_selection["Variable"].isin(worry_vars)
]

display(worry_features)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
168,Q142,Security Concerns and Victimization,Predictor candidate,Candidate,Security,214,int64,0,0.0,7,4443,-1: 1150; -2: 821; -5: 2472,2 | 3 | 1 | 4 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
169,Q143,Security Concerns and Victimization,Predictor candidate,Candidate,Security,215,int64,0,0.0,7,6856,-1: 1292; -2: 1369; -5: 4195,2 | 4 | 1 | 3 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
170,Q144,Security Concerns and Victimization,Predictor candidate,Candidate,Security,216,int64,0,0.0,6,3740,-1: 262; -2: 236; -4: 3056; -5: 186,2 | 1 | -2 | -1 | -4 | -5,Survey question / question-derived field,Binary,NaN,NaN,NaN,Review,
171,Q145,Security Concerns and Victimization,Predictor candidate,Candidate,Security,217,int64,0,0.0,6,4188,-1: 578; -2: 336; -4: 3056; -5: 218,2 | 1 | -1 | -2 | -4 | -5,Survey question / question-derived field,Binary,NaN,NaN,NaN,Review,
172,Q146,Security Concerns and Victimization,Predictor candidate,Candidate,Security,218,int64,0,0.0,8,3277,-1: 693; -2: 301; -4: 1200; -5: 1083,4 | 3 | 1 | 2 | -2 | -1 | -5 | -4,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
173,Q147,Security Concerns and Victimization,Predictor candidate,Candidate,Security,219,int64,0,0.0,8,2370,-1: 730; -2: 253; -4: 1200; -5: 187,2 | 4 | 1 | 3 | -2 | -1 | -5 | -4,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
174,Q148,Security Concerns and Victimization,Predictor candidate,Candidate,Security,220,int64,0,0.0,8,6894,-1: 862; -2: 314; -4: 5512; -5: 206,4 | 1 | 2 | 3 | -2 | -1 | -4 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,


In [61]:
worry_vars = [
    f"Q{i}" for i in range(142, 149)
]

worry_keep = [
    "Q142", "Q144", "Q145",
    "Q146", "Q147"
]

worry_optional = [
    "Q143", "Q148"
]

feature_selection.loc[
    feature_selection["Variable"].isin(worry_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(worry_keep),
    "Selection_reason"
] = (
    "Measures economic, personal or national security concerns "
    "and victimization; potentially relevant to life satisfaction."
)

feature_selection.loc[
    feature_selection["Variable"].isin(worry_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(worry_optional),
    "Selection_reason"
] = (
    "Potentially relevant security concern, but requires "
    "additional missingness and country coverage review."
)

In [62]:
display(
    feature_selection[
        feature_selection["Variable"].isin(worry_vars)
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
168,Q142,Ordinal,Keep,"Measures economic, personal or national securi..."
169,Q143,Ordinal,Optional,"Potentially relevant security concern, but req..."
170,Q144,Binary,Keep,"Measures economic, personal or national securi..."
171,Q145,Binary,Keep,"Measures economic, personal or national securi..."
172,Q146,Ordinal,Keep,"Measures economic, personal or national securi..."
173,Q147,Ordinal,Keep,"Measures economic, personal or national securi..."
174,Q148,Ordinal,Optional,"Potentially relevant security concern, but req..."


In [63]:
values_vars = [
    "Q149", "Q150", "Q151"
]

values_features = feature_selection[
    feature_selection["Variable"].isin(values_vars)
]

display(values_features)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
175,Q149,"Freedom, Equality and Political Values",Predictor candidate,Candidate,Political & Institutional,221,int64,0,0.0,5,2032,-1: 1412; -2: 400; -5: 220,1 | 2 | -2 | -1 | -5,Survey question / question-derived field,Binary,NaN,NaN,NaN,Review,
176,Q150,"Freedom, Equality and Political Values",Predictor candidate,Candidate,Political & Institutional,222,int64,0,0.0,5,1688,-1: 1058; -2: 433; -5: 197,2 | 1 | -2 | -1 | -5,Survey question / question-derived field,Binary,NaN,NaN,NaN,Review,
177,Q151,"Freedom, Equality and Political Values",Predictor candidate,Candidate,Political & Institutional,223,int64,0,0.0,6,8977,-1: 4473; -2: 934; -4: 3056; -5: 514,1 | 2 | -2 | -1 | -5 | -4,Survey question / question-derived field,Binary,NaN,NaN,NaN,Review,


In [64]:
values_keep = ["Q149", "Q150"]
values_optional = ["Q151"]

feature_selection.loc[
    feature_selection["Variable"].isin(values_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(values_keep),
    "Selection_reason"
] = (
    "Measures personal and political value priorities "
    "concerning freedom, equality and security."
)

feature_selection.loc[
    feature_selection["Variable"].isin(values_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(values_optional),
    "Selection_reason"
] = (
    "Measures willingness to fight for one's country; "
    "potentially relevant to national identity, but has "
    "relatively high missingness."
)

In [65]:
display(
    feature_selection[
        feature_selection["Variable"].isin(
            ["Q149", "Q150", "Q151"]
        )
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
175,Q149,Binary,Keep,Measures personal and political value prioriti...
176,Q150,Binary,Keep,Measures personal and political value prioriti...
177,Q151,Binary,Optional,Measures willingness to fight for one's countr...


In [66]:
postmaterialist_vars = [
    f"Q{i}" for i in range(152, 158)
]

display(
    feature_selection[
        feature_selection["Variable"].isin(
            postmaterialist_vars
        )
    ]
)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
178,Q152,Postmaterialist Priorities,Predictor candidate,Candidate,Personal Values & Wellbeing,224,int64,0,0.0,7,2557,-1: 1620; -2: 501; -5: 436,1 | 3 | 4 | 2 | -2 | -1 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,
179,Q153,Postmaterialist Priorities,Predictor candidate,Candidate,Personal Values & Wellbeing,225,int64,0,0.0,8,4877,-1: 957; -2: 1480; -3: 2388; -5: 52,4 | 3 | 2 | 1 | -3 | -2 | -1 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,
180,Q154,Postmaterialist Priorities,Predictor candidate,Candidate,Personal Values & Wellbeing,226,int64,0,0.0,7,1802,-1: 865; -2: 485; -5: 452,3 | 4 | 1 | 2 | -2 | -1 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,
181,Q155,Postmaterialist Priorities,Predictor candidate,Candidate,Personal Values & Wellbeing,227,int64,0,0.0,8,3777,-1: 833; -2: 1214; -3: 1619; -5: 111,1 | 2 | 4 | 3 | -3 | -2 | -1 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,
182,Q156,Postmaterialist Priorities,Predictor candidate,Candidate,Personal Values & Wellbeing,228,int64,0,0.0,7,1745,-1: 865; -2: 458; -5: 422,1 | 2 | 3 | 4 | -2 | -1 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,
183,Q157,Postmaterialist Priorities,Predictor candidate,Candidate,Personal Values & Wellbeing,229,int64,0,0.0,8,3626,-1: 808; -2: 1166; -3: 1529; -5: 123,3 | 4 | 2 | 1 | -2 | -3 | -1 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,


In [67]:
postmaterialist_vars = [
    f"Q{i}" for i in range(152, 158)
]

postmaterialist_keep = [
    "Q152", "Q154", "Q156"
]

postmaterialist_optional = [
    "Q153", "Q155", "Q157"
]

feature_selection.loc[
    feature_selection["Variable"].isin(postmaterialist_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(postmaterialist_keep),
    "Selection_reason"
] = (
    "Measures the respondent's first-choice priorities "
    "concerning national goals and social development."
)

feature_selection.loc[
    feature_selection["Variable"].isin(postmaterialist_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(postmaterialist_optional),
    "Selection_reason"
] = (
    "Measures second-choice priorities; provides additional "
    "information but may contain structural missingness "
    "when no first choice was provided."
)

In [68]:
display(
    feature_selection[
        feature_selection["Variable"].isin(postmaterialist_vars)
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
178,Q152,Nominal,Keep,Measures the respondent's first-choice priorit...
179,Q153,Nominal,Optional,Measures second-choice priorities; provides ad...
180,Q154,Nominal,Keep,Measures the respondent's first-choice priorit...
181,Q155,Nominal,Optional,Measures second-choice priorities; provides ad...
182,Q156,Nominal,Keep,Measures the respondent's first-choice priorit...
183,Q157,Nominal,Optional,Measures second-choice priorities; provides ad...


In [69]:
science_vars = [
    f"Q{i}" for i in range(158, 164)
]

display(
    feature_selection[
        feature_selection["Variable"].isin(science_vars)
    ]
)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
184,Q158,Science and Technology Attitudes,Predictor candidate,Candidate,Science & Technology,230,int64,0,0.0,14,2562,-1: 926; -2: 242; -4: 1190; -5: 204,7 | 9 | 5 | 10 | 8 | 6 | 4 | 3,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
185,Q159,Science and Technology Attitudes,Predictor candidate,Candidate,Science & Technology,231,int64,0,0.0,14,2820,-1: 1168; -2: 276; -4: 1190; -5: 186,6 | 9 | 7 | 10 | 5 | 8 | 4 | 1,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
186,Q160,Science and Technology Attitudes,Predictor candidate,Candidate,Science & Technology,232,int64,0,0.0,14,3807,-1: 1969; -2: 449; -4: 1190; -5: 199,7 | 9 | 8 | 5 | 6 | 10 | 4 | 2,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
187,Q161,Science and Technology Attitudes,Predictor candidate,Candidate,Science & Technology,233,int64,0,0.0,14,5035,-1: 3079; -2: 553; -4: 1190; -5: 213,5 | 2 | 6 | 10 | 4 | 8 | 3 | 9,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
188,Q162,Science and Technology Attitudes,Predictor candidate,Candidate,Science & Technology,234,int64,0,0.0,14,3529,-1: 1745; -2: 392; -4: 1190; -5: 202,2 | 4 | 3 | 10 | 8 | 1 | 5 | 7,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
189,Q163,Science and Technology Attitudes,Predictor candidate,Candidate,Science & Technology,235,int64,0,0.0,14,2711,-1: 1057; -2: 302; -4: 1190; -5: 162,6 | 10 | 5 | 7 | 9 | 8 | 2 | 4,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,


In [70]:
science_vars = [
    f"Q{i}" for i in range(158, 164)
]

feature_selection.loc[
    feature_selection["Variable"].isin(science_vars),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(science_vars),
    "Selection_reason"
] = (
    "Measures attitudes towards science and technology, "
    "including perceived benefits, risks and societal importance."
)

In [71]:
display(
    feature_selection[
        feature_selection["Variable"].isin(science_vars)
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
184,Q158,Ordinal,Keep,Measures attitudes towards science and technol...
185,Q159,Ordinal,Keep,Measures attitudes towards science and technol...
186,Q160,Ordinal,Keep,Measures attitudes towards science and technol...
187,Q161,Ordinal,Keep,Measures attitudes towards science and technol...
188,Q162,Ordinal,Keep,Measures attitudes towards science and technol...
189,Q163,Ordinal,Keep,Measures attitudes towards science and technol...


In [72]:
religious_vars = [
    f"Q{i}" for i in range(164, 176)
]

display(
    feature_selection[
        feature_selection["Variable"].isin(religious_vars)
    ]
)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
190,Q164,Religious Beliefs,Predictor candidate,Candidate,Personal Values & Wellbeing,236,int64,0,0.0,14,2253,-1: 490; -2: 356; -4: 1200; -5: 207,7 | 1 | 8 | 4 | 3 | 5 | 10 | 6,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
191,Q165,Religious Beliefs,Predictor candidate,Candidate,Personal Values & Wellbeing,237,int64,0,0.0,6,4691,-1: 1607; -2: 465; -4: 2400; -5: 219,1 | 2 | -2 | -1 | -5 | -4,Survey question / question-derived field,Binary,NaN,NaN,NaN,Review,
192,Q166,Religious Beliefs,Predictor candidate,Candidate,Personal Values & Wellbeing,238,int64,0,0.0,6,7595,-1: 4205; -2: 776; -4: 2400; -5: 214,1 | 2 | -2 | -1 | -5 | -4,Survey question / question-derived field,Binary,NaN,NaN,NaN,Review,
193,Q167,Religious Beliefs,Predictor candidate,Candidate,Personal Values & Wellbeing,239,int64,0,0.0,6,7144,-1: 3694; -2: 805; -4: 2400; -5: 245,2 | 1 | -2 | -1 | -5 | -4,Survey question / question-derived field,Binary,NaN,NaN,NaN,Review,
194,Q168,Religious Beliefs,Predictor candidate,Candidate,Personal Values & Wellbeing,240,int64,0,0.0,6,6971,-1: 3567; -2: 761; -4: 2400; -5: 243,1 | 2 | -2 | -1 | -5 | -4,Survey question / question-derived field,Binary,NaN,NaN,NaN,Review,
195,Q169,Religious Beliefs,Predictor candidate,Candidate,Personal Values & Wellbeing,241,int64,0,0.0,7,5593,-1: 4079; -2: 1270; -5: 244,3 | 1 | 4 | 2 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
196,Q170,Religious Beliefs,Predictor candidate,Candidate,Personal Values & Wellbeing,242,int64,0,0.0,8,7319,-1: 2777; -2: 2000; -4: 1200; -5: 1342,1 | 3 | 4 | 2 | -2 | -1 | -5 | -4,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
197,Q171,Religious Practices and Identity,Predictor candidate,Candidate,Personal Values & Wellbeing,243,int64,0,0.0,10,1212,-1: 374; -2: 512; -5: 326,7 | 3 | 6 | 2 | 4 | 5 | 1 | -2,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
198,Q172,Religious Practices and Identity,Predictor candidate,Candidate,Personal Values & Wellbeing,244,int64,0,0.0,11,1495,-1: 628; -2: 624; -5: 243,6 | 8 | 2 | 5 | 7 | 4 | 3 | 1,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
200,Q173,Religious Practices and Identity,Predictor candidate,Candidate,Personal Values & Wellbeing,246,int64,0,0.0,6,2652,-1: 1565; -2: 821; -5: 266,1 | 2 | 3 | -2 | -1 | -5,Survey question / question-derived field,Nominal,NaN,NaN,NaN,Review,


In [73]:
religious_keep = [
    "Q164", "Q165", "Q169",
    "Q171", "Q172", "Q173"
]

religious_optional = [
    "Q166", "Q167", "Q168",
    "Q170", "Q174", "Q175", "Q172R"
]

feature_selection.loc[
    feature_selection["Variable"].isin(religious_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(religious_keep),
    "Selection_reason"
] = (
    "Measures religious beliefs, identity, practices or "
    "the importance of religion in personal life."
)

feature_selection.loc[
    feature_selection["Variable"].isin(religious_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(religious_optional),
    "Selection_reason"
] = (
    "Potentially relevant religious belief or attitude predictor; "
    "requires additional missingness, coverage or redundancy review."
)

In [74]:
religious_check = [
    f"Q{i}" for i in range(164, 176)
] + ["Q172R"]

display(
    feature_selection[
        feature_selection["Variable"].isin(religious_check)
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
190,Q164,Ordinal,Keep,"Measures religious beliefs, identity, practice..."
191,Q165,Binary,Keep,"Measures religious beliefs, identity, practice..."
192,Q166,Binary,Optional,Potentially relevant religious belief or attit...
193,Q167,Binary,Optional,Potentially relevant religious belief or attit...
194,Q168,Binary,Optional,Potentially relevant religious belief or attit...
195,Q169,Ordinal,Keep,"Measures religious beliefs, identity, practice..."
196,Q170,Ordinal,Optional,Potentially relevant religious belief or attit...
197,Q171,Ordinal,Keep,"Measures religious beliefs, identity, practice..."
198,Q172,Ordinal,Keep,"Measures religious beliefs, identity, practice..."
199,Q172R,Ordinal,Optional,Potentially relevant religious belief or attit...


In [75]:
ethical_vars = [
    f"Q{i}" for i in range(176, 196)
]

display(
    feature_selection[
        feature_selection["Variable"].isin(ethical_vars)
    ]
)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
203,Q176,Ethical Values and Moral Norms,Predictor candidate,Candidate,Personal Values & Wellbeing,249,int64,0,0.0,13,3391,-1: 2574; -2: 641; -5: 176,4 | -2 | 5 | 10 | 6 | 7 | 1 | 8,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
204,Q177,Ethical Values and Moral Norms,Predictor candidate,Candidate,Personal Values & Wellbeing,250,int64,0,0.0,13,1688,-1: 1082; -2: 410; -5: 196,1 | 10 | 9 | 3 | -2 | 5 | 6 | 4,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
205,Q178,Ethical Values and Moral Norms,Predictor candidate,Candidate,Personal Values & Wellbeing,251,int64,0,0.0,14,3539,-1: 601; -2: 352; -4: 2403; -5: 183,1 | 10 | 5 | 7 | 2 | -2 | 8 | 3,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
206,Q179,Ethical Values and Moral Norms,Predictor candidate,Candidate,Personal Values & Wellbeing,252,int64,0,0.0,13,905,-1: 364; -2: 360; -5: 181,1 | 3 | 2 | 5 | 6 | 4 | 10 | 7,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
207,Q180,Ethical Values and Moral Norms,Predictor candidate,Candidate,Personal Values & Wellbeing,253,int64,0,0.0,13,1279,-1: 714; -2: 394; -5: 171,1 | 5 | 3 | 6 | 7 | 2 | 4 | 10,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
208,Q181,Ethical Values and Moral Norms,Predictor candidate,Candidate,Personal Values & Wellbeing,254,int64,0,0.0,13,1194,-1: 641; -2: 364; -5: 189,1 | 5 | 2 | 4 | 3 | 10 | 8 | 6,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
209,Q182,Ethical Values and Moral Norms,Predictor candidate,Candidate,Personal Values & Wellbeing,255,int64,0,0.0,14,7515,-1: 2524; -2: 1121; -4: 3650; -5: 220,4 | 9 | 10 | 6 | 8 | 7 | 1 | 5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
210,Q183,Ethical Values and Moral Norms,Predictor candidate,Candidate,Personal Values & Wellbeing,256,int64,0,0.0,14,8751,-1: 1636; -2: 595; -4: 6302; -5: 218,3 | 5 | 6 | 1 | 8 | 10 | 4 | 7,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
211,Q184,Ethical Values and Moral Norms,Predictor candidate,Candidate,Personal Values & Wellbeing,257,int64,0,0.0,13,2275,-1: 1461; -2: 614; -5: 200,2 | 9 | 5 | 4 | 1 | 10 | 6 | 8,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
212,Q185,Ethical Values and Moral Norms,Predictor candidate,Candidate,Personal Values & Wellbeing,258,int64,0,0.0,13,1929,-1: 1228; -2: 499; -5: 202,5 | 9 | 6 | 7 | 10 | -2 | 8 | 1,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,


In [76]:
ethical_vars = [f"Q{i}" for i in range(176, 196)]

ethical_keep = [
    f"Q{i}" for i in range(176, 192)
] + [
    "Q193", "Q194", "Q195"
]

feature_selection.loc[
    feature_selection["Variable"].isin(ethical_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(ethical_keep),
    "Selection_reason"
] = (
    "Measures ethical attitudes and moral judgements "
    "about social behaviour and controversial issues."
)

feature_selection.loc[
    feature_selection["Variable"] == "Q192",
    "Selection_decision"
] = "Review"

feature_selection.loc[
    feature_selection["Variable"] == "Q192",
    "Selection_reason"
] = "Review exact variable label in the WVS codebook."

In [77]:
display(
    feature_selection[
        feature_selection["Variable"].isin(ethical_vars)
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
203,Q176,Ordinal,Keep,Measures ethical attitudes and moral judgement...
204,Q177,Ordinal,Keep,Measures ethical attitudes and moral judgement...
205,Q178,Ordinal,Keep,Measures ethical attitudes and moral judgement...
206,Q179,Ordinal,Keep,Measures ethical attitudes and moral judgement...
207,Q180,Ordinal,Keep,Measures ethical attitudes and moral judgement...
208,Q181,Ordinal,Keep,Measures ethical attitudes and moral judgement...
209,Q182,Ordinal,Keep,Measures ethical attitudes and moral judgement...
210,Q183,Ordinal,Keep,Measures ethical attitudes and moral judgement...
211,Q184,Ordinal,Keep,Measures ethical attitudes and moral judgement...
212,Q185,Ordinal,Keep,Measures ethical attitudes and moral judgement...


In [78]:
political_vars = [
    f"Q{i}" for i in range(199, 235)
]

display(
    feature_selection[
        feature_selection["Variable"].isin(political_vars)
    ]
)

,Variable,Feature_group,Variable_role,Model_status,Category,Column number,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category,Measurement_type,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Selection_decision,Selection_reason
226,Q199,Political Interest and Discussion,Predictor candidate,Candidate,Political & Institutional,272,int64,0,0.0,7,731,-1: 316; -2: 251; -5: 164,2 | 4 | 3 | 1 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
227,Q200,Political Interest and Discussion,Predictor candidate,Candidate,Political & Institutional,273,int64,0,0.0,6,987,-1: 468; -2: 335; -5: 184,2 | -2 | 3 | 1 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
228,Q201,News and Information Sources,Predictor candidate,Candidate,Social & Community,274,int64,0,0.0,8,928,-1: 423; -2: 289; -5: 216,2 | 1 | 5 | 4 | 3 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
229,Q202,News and Information Sources,Predictor candidate,Candidate,Social & Community,275,int64,0,0.0,9,602,-1: 198; -2: 225; -3: 11; -5: 168,2 | 1 | 5 | 3 | 4 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
230,Q203,News and Information Sources,Predictor candidate,Candidate,Social & Community,276,int64,0,0.0,8,1074,-1: 497; -2: 338; -5: 239,1 | 5 | 2 | 3 | 4 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
231,Q204,News and Information Sources,Predictor candidate,Candidate,Social & Community,277,int64,0,0.0,8,995,-1: 389; -2: 345; -5: 261,5 | 1 | 2 | 4 | 3 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
232,Q205,News and Information Sources,Predictor candidate,Candidate,Social & Community,278,int64,0,0.0,8,1964,-1: 920; -2: 439; -5: 605,5 | 1 | 2 | 4 | 3 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
233,Q206,News and Information Sources,Predictor candidate,Candidate,Social & Community,279,int64,0,0.0,8,1626,-1: 730; -2: 374; -5: 522,5 | 1 | 2 | 4 | 3 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
234,Q207,News and Information Sources,Predictor candidate,Candidate,Social & Community,280,int64,0,0.0,9,3648,-1: 739; -2: 315; -4: 1813; -5: 781,5 | 1 | 2 | 4 | 3 | -2 | -5 | -1,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,
235,Q208,News and Information Sources,Predictor candidate,Candidate,Social & Community,281,int64,0,0.0,8,1085,-1: 548; -2: 310; -5: 227,2 | 1 | 4 | 5 | 3 | -2 | -1 | -5,Survey question / question-derived field,Ordinal,NaN,NaN,NaN,Review,


In [79]:
voting_vars = ["Q221", "Q222"]

for df in [master_dictionary, feature_selection]:
    df.loc[
        df["Variable"].isin(voting_vars),
        "Measurement_type"
    ] = "Nominal"

In [80]:
# Keep
political_keep = (
    [f"Q{i}" for i in range(199, 217)]
    + ["Q221", "Q222"]
    + [f"Q{i}" for i in range(224, 235)]
)

# Optional
political_optional = (
    [f"Q{i}" for i in range(217, 221)]
    + ["Q223"]
)

feature_selection.loc[
    feature_selection["Variable"].isin(political_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(political_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(political_keep),
    "Selection_reason"
] = (
    "Measures political interest, information exposure, "
    "political participation, voting behaviour or perceived "
    "electoral integrity."
)

feature_selection.loc[
    feature_selection["Variable"].isin(political_optional),
    "Selection_reason"
] = (
    "Potentially relevant political or digital participation "
    "measure, but may have structural missingness or "
    "country-specific comparability limitations."
)

In [81]:
political_vars = [f"Q{i}" for i in range(199, 235)]

display(
    feature_selection[
        feature_selection["Variable"].isin(political_vars)
    ][
        [
            "Variable",
            "Measurement_type",
            "Selection_decision",
            "Selection_reason"
        ]
    ]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
226,Q199,Ordinal,Keep,"Measures political interest, information expos..."
227,Q200,Ordinal,Keep,"Measures political interest, information expos..."
228,Q201,Ordinal,Keep,"Measures political interest, information expos..."
229,Q202,Ordinal,Keep,"Measures political interest, information expos..."
230,Q203,Ordinal,Keep,"Measures political interest, information expos..."
231,Q204,Ordinal,Keep,"Measures political interest, information expos..."
232,Q205,Ordinal,Keep,"Measures political interest, information expos..."
233,Q206,Ordinal,Keep,"Measures political interest, information expos..."
234,Q207,Ordinal,Keep,"Measures political interest, information expos..."
235,Q208,Ordinal,Keep,"Measures political interest, information expos..."


In [82]:
political_culture_vars = [f"Q{i}" for i in range(235, 260)]

display(
    feature_selection[
        feature_selection["Variable"].isin(political_culture_vars)
    ][[
        "Variable",
        "Measurement_type",
        "Selection_decision",
        "Selection_reason"
    ]]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
265,Q235,Ordinal,Review,
266,Q236,Ordinal,Review,
267,Q237,Ordinal,Review,
268,Q238,Ordinal,Review,
269,Q239,Ordinal,Review,
270,Q240,Ordinal,Review,
271,Q241,Ordinal,Review,
272,Q242,Ordinal,Review,
273,Q243,Ordinal,Review,
274,Q244,Ordinal,Review,


In [83]:
political_culture_vars = [f"Q{i}" for i in range(235, 260)]

# All variables are retained as predictor candidates
feature_selection.loc[
    feature_selection["Variable"].isin(political_culture_vars),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(political_culture_vars),
    "Selection_reason"
] = (
    "Measures political culture, democratic values, "
    "political system evaluations, national pride "
    "or social and national belonging."
)

# Q254 is nominal because response category 5
# does not represent an ordered level of national pride
for df in [master_dictionary, feature_selection]:
    df.loc[
        df["Variable"] == "Q254",
        "Measurement_type"
    ] = "Nominal"

In [84]:
display(
    feature_selection[
        feature_selection["Variable"].isin(political_culture_vars)
    ][[
        "Variable",
        "Measurement_type",
        "Selection_decision",
        "Selection_reason"
    ]]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
265,Q235,Ordinal,Keep,"Measures political culture, democratic values,..."
266,Q236,Ordinal,Keep,"Measures political culture, democratic values,..."
267,Q237,Ordinal,Keep,"Measures political culture, democratic values,..."
268,Q238,Ordinal,Keep,"Measures political culture, democratic values,..."
269,Q239,Ordinal,Keep,"Measures political culture, democratic values,..."
270,Q240,Ordinal,Keep,"Measures political culture, democratic values,..."
271,Q241,Ordinal,Keep,"Measures political culture, democratic values,..."
272,Q242,Ordinal,Keep,"Measures political culture, democratic values,..."
273,Q243,Ordinal,Keep,"Measures political culture, democratic values,..."
274,Q244,Ordinal,Keep,"Measures political culture, democratic values,..."


In [85]:
demographic_vars = [
    f"Q{i}" for i in range(260, 291)
]

display(
    feature_selection[
        feature_selection["Variable"].isin(demographic_vars)
    ][[
        "Variable",
        "Measurement_type",
        "Selection_decision",
        "Selection_reason"
    ]]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
290,Q260,Nominal,Keep,Relevant demographic background predictor.
291,Q261,Numeric,Exclude,Redundant with Q262; use respondent age Q262 i...
292,Q262,Numeric,Keep,Relevant demographic background predictor.
293,Q263,Binary,Keep,Relevant demographic background predictor.
294,Q264,Binary,Optional,Potentially useful family or migration backgro...
295,Q265,Binary,Optional,Potentially useful family or migration backgro...
296,Q266,Nominal,Optional,Potentially useful family or migration backgro...
297,Q267,Nominal,Optional,Potentially useful family or migration backgro...
298,Q268,Nominal,Optional,Potentially useful family or migration backgro...
299,Q269,Nominal,Review,Requires further review of coding and cross-co...


In [86]:
# Correct measurement types
# Correct measurement types
measurement_updates = {
    "Q270": "Numeric",
    "Q274": "Numeric",
}

for variable, measurement_type in measurement_updates.items():
    for df in [master_dictionary, feature_selection]:
        df.loc[
            df["Variable"] == variable,
            "Measurement_type"
        ] = measurement_type


# Update selection decisions
decision_updates = {
    "Q269": "Keep",
    "Q272": "Optional",
    "Q289": "Optional",
    "Q290": "Exclude"
}

for variable, decision in decision_updates.items():
    feature_selection.loc[
        feature_selection["Variable"] == variable,
        "Selection_decision"
    ] = decision


# Update reasons
reason_updates = {
    "Q269": "Measures respondent citizenship status, including a country-specific pending application category.",
    "Q272": "Measures language spoken at home; potentially relevant, but has many categories and possible country-level confounding.",
    "Q289": "Measures major religious affiliation categories; potentially relevant, but may reflect substantial cross-country differences.",
    "Q290": "Measures country-specific racial or ethnic categories with limited cross-country comparability."
}

for variable, reason in reason_updates.items():
    feature_selection.loc[
        feature_selection["Variable"] == variable,
        "Selection_reason"
    ] = reason

In [87]:
demographic_vars = [
    f"Q{i}" for i in range(260, 291)
]

display(
    feature_selection[
        feature_selection["Variable"].isin(demographic_vars)
    ][[
        "Variable",
        "Measurement_type",
        "Selection_decision",
        "Selection_reason"
    ]]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
290,Q260,Nominal,Keep,Relevant demographic background predictor.
291,Q261,Numeric,Exclude,Redundant with Q262; use respondent age Q262 i...
292,Q262,Numeric,Keep,Relevant demographic background predictor.
293,Q263,Binary,Keep,Relevant demographic background predictor.
294,Q264,Binary,Optional,Potentially useful family or migration backgro...
295,Q265,Binary,Optional,Potentially useful family or migration backgro...
296,Q266,Nominal,Optional,Potentially useful family or migration backgro...
297,Q267,Nominal,Optional,Potentially useful family or migration backgro...
298,Q268,Nominal,Optional,Potentially useful family or migration backgro...
299,Q269,Nominal,Keep,"Measures respondent citizenship status, includ..."


In [88]:
# Count variables by selection decision
display(
    feature_selection["Selection_decision"]
    .value_counts(dropna=False)
    .to_frame("Number of variables")
)

# Count variables by decision and category
display(
    feature_selection.groupby(
        ["Category", "Selection_decision"],
        dropna=False
    ).size().unstack(fill_value=0)
)

,Number of variables
Selection_decision,
Keep,233
Review,87
Optional,45
Exclude,7


Selection_decision,Exclude,Keep,Optional,Review
Category,,,,
Demographic & Background,6,15,14,1
Economic,0,17,1,0
Personal Values & Wellbeing,0,43,11,1
Political & Institutional,0,77,7,70
Science & Technology,0,6,0,0
Security,0,13,5,0
Social & Community,0,62,7,15
Target Variable,1,0,0,0


In [89]:
# Variables still requiring review
remaining_review = feature_selection[
    feature_selection["Selection_decision"].isin(
        ["Review"]
    )
]

display(
    remaining_review[[
        "Variable",
        "Category",
        "Measurement_type",
        "Missing percentage (NaN)",
        "Selection_reason"
    ]]
)

,Variable,Category,Measurement_type,Missing percentage (NaN),Selection_reason
80,Q83,Political & Institutional,Ordinal,0.0,
81,Q84,Political & Institutional,Ordinal,0.0,
82,Q85,Political & Institutional,Ordinal,0.0,
83,Q86,Political & Institutional,Ordinal,0.0,
84,Q87,Political & Institutional,Ordinal,0.0,
...,...,...,...,...,...
365,Q294A,Political & Institutional,Ordinal,0.0,
366,Q294B,Political & Institutional,Ordinal,0.0,
367,Q33_3,Social & Community,Ordinal,0.0,
368,Q34_3,Social & Community,Ordinal,0.0,


In [90]:
# Check missing selection decisions
display(
    feature_selection[
        feature_selection["Selection_decision"].isna()
    ][[
        "Variable",
        "Category",
        "Measurement_type"
    ]]
)

,Variable,Category,Measurement_type


In [91]:
# Show all variables requiring review
review_vars = feature_selection[
    feature_selection["Selection_decision"] == "Review"
]

print("Total review variables:", len(review_vars))

print(
    review_vars[
        review_vars["Category"] == "Political & Institutional"
    ][[
        "Variable",
        "Measurement_type",
        "Missing percentage (NaN)",
        "Candidate special-code details",
        "Selection_reason"
    ]].to_string(index=False)
)

Total review variables: 87
        Variable Measurement_type  Missing percentage (NaN)                   Candidate special-code details Selection_reason
             Q83          Ordinal                       0.0             -1: 9667; -2: 804; -4: 3036; -5: 314                 
             Q84          Ordinal                       0.0           -1: 14120; -2: 1000; -4: 3036; -5: 607                 
             Q85          Ordinal                       0.0           -1: 14926; -2: 1002; -4: 3036; -5: 659                 
             Q86          Ordinal                       0.0           -1: 16592; -2: 1181; -4: 6516; -5: 890                 
             Q87          Ordinal                       0.0            -1: 13052; -2: 938; -4: 4436; -5: 799                 
             Q88          Ordinal                       0.0            -1: 10149; -2: 715; -4: 4436; -5: 549                 
             Q89          Ordinal                       0.0            -1: 15010; -2: 983; 

In [92]:
print(
    review_vars[
        review_vars["Category"] == "Social & Community"
    ][[
        "Variable",
        "Measurement_type",
        "Missing percentage (NaN)",
        "Candidate special-code details",
        "Selection_reason"
    ]].to_string(index=False)
)

Variable Measurement_type  Missing percentage (NaN) Candidate special-code details Selection_reason
    Q94R           Binary                       0.0      -1: 583; -2: 320; -5: 191                 
    Q95R           Binary                       0.0      -1: 645; -2: 260; -5: 233                 
    Q96R           Binary                       0.0      -1: 719; -2: 258; -5: 237                 
    Q97R           Binary                       0.0      -1: 845; -2: 290; -5: 256                 
    Q98R           Binary                       0.0      -1: 800; -2: 314; -5: 230                 
    Q99R           Binary                       0.0      -1: 898; -2: 322; -5: 206                 
   Q100R           Binary                       0.0      -1: 977; -2: 347; -5: 273                 
   Q101R           Binary                       0.0      -1: 847; -2: 325; -5: 231                 
   Q102R           Binary                       0.0     -1: 1077; -2: 369; -5: 269                 


In [93]:
print(
    review_vars[
        ~review_vars["Category"].isin([
            "Political & Institutional",
            "Social & Community"
        ])
    ][[
        "Variable",
        "Category",
        "Measurement_type",
        "Selection_reason"
    ]].to_string(index=False)
)

Variable                    Category Measurement_type                                 Selection_reason
    Q192 Personal Values & Wellbeing          Ordinal Review exact variable label in the WVS codebook.
 Q289CS9    Demographic & Background          Nominal                                                 


In [94]:
# Political and Institutional variables

# Keep: international organization confidence
political_keep = (
    [f"Q{i}" for i in range(83, 94)]
    + [f"Q{i}" for i in range(196, 199)]
)

# Optional: country-specific organizations and political trust module
political_optional = (
    [f"Q82_{name}" for name in [
        "AFRICANUNION", "APEC", "ARABLEAGUE", "ASEAN",
        "CIS", "CUSMA", "ECO", "EU", "GULFCOOP",
        "ISLCOOP", "MERCOSUR", "NAFTA", "OAS",
        "SAARC", "SCO", "TLC", "UNDP"
    ]]
    + ["Q234A"]
    + [
        f"Q291{group}{i}"
        for group in ["P", "G", "UN"]
        for i in range(1, 7)
    ]
    + [f"Q292{letter}" for letter in "ABCDEFGHIJKLMNO"]
    + ["Q293", "Q294A", "Q294B"]
)

# Exclude: alternative text versions of party preference
political_exclude = ["Q223_ABREV", "Q223_LOCAL"]

feature_selection.loc[
    feature_selection["Variable"].isin(political_keep),
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"].isin(political_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(political_exclude),
    "Selection_decision"
] = "Exclude"

# Correct Q294 measurement types
for df in [master_dictionary, feature_selection]:
    df.loc[
        df["Variable"].isin(["Q294A", "Q294B"]),
        "Measurement_type"
    ] = "Ordinal"

In [95]:
check_vars = (
    political_keep
    + political_optional
    + political_exclude
)

display(
    feature_selection[
        feature_selection["Variable"].isin(check_vars)
    ][[
        "Variable",
        "Measurement_type",
        "Selection_decision"
    ]].to_string(index=False)
)

'        Variable Measurement_type Selection_decision\n             Q83          Ordinal               Keep\n             Q84          Ordinal               Keep\n             Q85          Ordinal               Keep\n             Q86          Ordinal               Keep\n             Q87          Ordinal               Keep\n             Q88          Ordinal               Keep\n             Q89          Ordinal               Keep\nQ82_AFRICANUNION          Ordinal           Optional\n        Q82_APEC          Ordinal           Optional\n  Q82_ARABLEAGUE          Ordinal           Optional\n       Q82_ASEAN          Ordinal           Optional\n         Q82_CIS          Ordinal           Optional\n       Q82_CUSMA          Ordinal           Optional\n         Q82_ECO          Ordinal           Optional\n          Q82_EU          Ordinal           Optional\n    Q82_GULFCOOP          Ordinal           Optional\n     Q82_ISLCOOP          Ordinal           Optional\n    Q82_MERCOSUR          O

In [96]:
membership_r_vars = [
    f"Q{i}R" for i in range(94, 106)
]

display(
    feature_selection[
        feature_selection["Variable"].isin(membership_r_vars)
    ][[
        "Variable",
        "Measurement_type",
        "Selection_decision",
        "Missing percentage (NaN)",
        "Candidate special-code details"
    ]].to_string(index=False)
)

'Variable Measurement_type Selection_decision  Missing percentage (NaN) Candidate special-code details\n    Q94R           Binary             Review                       0.0      -1: 583; -2: 320; -5: 191\n    Q95R           Binary             Review                       0.0      -1: 645; -2: 260; -5: 233\n    Q96R           Binary             Review                       0.0      -1: 719; -2: 258; -5: 237\n    Q97R           Binary             Review                       0.0      -1: 845; -2: 290; -5: 256\n    Q98R           Binary             Review                       0.0      -1: 800; -2: 314; -5: 230\n    Q99R           Binary             Review                       0.0      -1: 898; -2: 322; -5: 206\n   Q100R           Binary             Review                       0.0      -1: 977; -2: 347; -5: 273\n   Q101R           Binary             Review                       0.0      -1: 847; -2: 325; -5: 231\n   Q102R           Binary             Review                       0.0  

In [97]:
# Original binary recoded membership variables
membership_exclude = [
    f"Q{i}R" for i in range(94, 104)
]

# Alternative versions with higher missingness
membership_optional = ["Q104R", "Q105R"]

feature_selection.loc[
    feature_selection["Variable"].isin(membership_exclude),
    "Selection_decision"
] = "Exclude"

feature_selection.loc[
    feature_selection["Variable"].isin(membership_optional),
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"].isin(membership_exclude),
    "Selection_reason"
] = (
    "Binary recoded version of an original organizational "
    "membership variable; excluded to avoid redundancy "
    "with the original variable."
)

feature_selection.loc[
    feature_selection["Variable"].isin(membership_optional),
    "Selection_reason"
] = (
    "Alternative binary recoded version of an organizational "
    "membership variable; retained as an optional alternative "
    "because the original variable has higher missingness."
)

In [98]:
display(
    feature_selection[
        feature_selection["Variable"].isin(
            [f"Q{i}R" for i in range(94, 106)]
        )
    ][[
        "Variable",
        "Measurement_type",
        "Selection_decision",
        "Selection_reason"
    ]].to_string(index=False)
)

'Variable Measurement_type Selection_decision                                                                                                                                                       Selection_reason\n    Q94R           Binary            Exclude                                     Binary recoded version of an original organizational membership variable; excluded to avoid redundancy with the original variable.\n    Q95R           Binary            Exclude                                     Binary recoded version of an original organizational membership variable; excluded to avoid redundancy with the original variable.\n    Q96R           Binary            Exclude                                     Binary recoded version of an original organizational membership variable; excluded to avoid redundancy with the original variable.\n    Q97R           Binary            Exclude                                     Binary recoded version of an original organizational membership va

In [99]:
extra_vars = ["Q33_3", "Q34_3", "Q35_3"]

display(
    feature_selection[
        feature_selection["Variable"].isin(extra_vars)
    ][[
        "Variable",
        "Measurement_type",
        "Selection_decision",
        "Example values",
        "Candidate special-code details"
    ]].to_string(index=False)
)

'Variable Measurement_type Selection_decision           Example values Candidate special-code details\n   Q33_3          Ordinal             Review 2 | 3 | 1 | -2 | -1 | -5      -1: 503; -2: 122; -5: 111\n   Q34_3          Ordinal             Review 1 | 2 | 3 | -2 | -1 | -5      -1: 626; -2: 177; -5: 110\n   Q35_3          Ordinal             Review 3 | 2 | 1 | -2 | -1 | -5     -1: 1140; -2: 199; -5: 125'

In [100]:
# Alternative three-category versions
alternative_vars = ["Q33_3", "Q34_3", "Q35_3"]

feature_selection.loc[
    feature_selection["Variable"].isin(alternative_vars),
    "Selection_decision"
] = "Exclude"

feature_selection.loc[
    feature_selection["Variable"].isin(alternative_vars),
    "Selection_reason"
] = (
    "Three-category alternative version of an original "
    "five-category attitude variable; excluded from the "
    "main model to avoid redundant representations."
)

In [101]:
display(
    feature_selection[
        feature_selection["Variable"] == "Q289CS9"
    ][[
        "Variable",
        "Measurement_type",
        "Selection_decision",
        "Example values",
        "Candidate special-code details"
    ]]
)

,Variable,Measurement_type,Selection_decision,Example values,Candidate special-code details
329,Q289CS9,Nominal,Review,10100000 | 100000020 | 20000000 | 30100000 | 9...,-1: 336; -2: 765; -4: 9; -5: 255


In [102]:
feature_selection.loc[
    feature_selection["Variable"] == "Q289CS9",
    "Selection_decision"
] = "Optional"

feature_selection.loc[
    feature_selection["Variable"] == "Q289CS9",
    "Selection_reason"
] = (
    "Detailed country-specific religious denomination; "
    "retained for exploratory analysis because it provides "
    "more detailed information than the cross-country "
    "religious category variable Q289."
)

In [103]:
remaining_review = feature_selection[
    feature_selection["Selection_decision"] == "Review"
]

print("Remaining Review variables:", len(remaining_review))

display(
    remaining_review[[
        "Variable",
        "Category",
        "Measurement_type",
        "Selection_reason"
    ]].to_string(index=False)
)

Remaining Review variables: 1


'Variable                    Category Measurement_type                                 Selection_reason\n    Q192 Personal Values & Wellbeing          Ordinal Review exact variable label in the WVS codebook.'

In [104]:
feature_selection.loc[
    feature_selection["Variable"] == "Q192",
    "Selection_decision"
] = "Keep"

feature_selection.loc[
    feature_selection["Variable"] == "Q192",
    "Selection_reason"
] = (
    "Measures the respondent's moral judgement of "
    "whether terrorism can be justified; relevant "
    "to ethical values and moral norms."
)

display(
    feature_selection[
        feature_selection["Variable"] == "Q192"
    ][[
        "Variable",
        "Measurement_type",
        "Selection_decision",
        "Selection_reason"
    ]]
)

,Variable,Measurement_type,Selection_decision,Selection_reason
219,Q192,Ordinal,Keep,Measures the respondent's moral judgement of w...


In [105]:
# 1. Count all selection decisions
print("Selection decision summary:")
display(
    feature_selection["Selection_decision"]
    .value_counts(dropna=False)
)

# 2. Check whether any Review variables remain
review_variables = feature_selection.loc[
    feature_selection["Selection_decision"] == "Review",
    ["Variable", "Feature_group", "Measurement_type",
     "Selection_reason"]
]

print("Remaining Review variables:", len(review_variables))
display(review_variables)

# 3. Check missing selection decisions
print(
    "Missing selection decisions:",
    feature_selection["Selection_decision"].isna().sum()
)

Selection decision summary:


Selection_decision
Keep        248
Optional    102
Exclude      22
Name: count, dtype: int64

Remaining Review variables: 0


,Variable,Feature_group,Measurement_type,Selection_reason


Missing selection decisions: 0


| Selection decision | Variables | Percentage |
|---|---:|---:|
| Keep | 248 | 66.67% |
| Optional | 102 | 27.42% |
| Exclude | 22 | 5.91% |
| Review | 0 | 0% |
| **Total** | **372** | **100%** |

In [106]:
# Save the final feature selection and variable dictionary
# This is the single authoritative save point, after all selection updates.

master_dictionary.to_csv(
    output_dir / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

feature_selection.to_csv(
    output_dir / "WVS_Feature_Selection.csv",
    index=False
)

print("Final files saved to:", output_dir.resolve())


Final files saved to: C:\Users\aiqiu\Documents\Nackademin\ML\wvs_mlproject\outputs\feature_selection


In [109]:
# Read back and validate the final saved files
feature_check = pd.read_csv(
    output_dir / "WVS_Feature_Selection.csv"
)
dictionary_check = pd.read_csv(
    output_dir / "WVS_Master_Variable_Dictionary.csv"
)

# Basic structure
assert feature_check.shape == (372, 20), feature_check.shape
assert dictionary_check.shape == (372, 18), dictionary_check.shape
assert feature_check["Variable"].is_unique
assert dictionary_check["Variable"].is_unique

# Expected final decision counts from the reviewed selection summary
expected_counts = {"Keep": 248, "Optional": 102, "Exclude": 22}
actual_counts = feature_check["Selection_decision"].value_counts().to_dict()
assert actual_counts == expected_counts, actual_counts
assert feature_check["Selection_decision"].notna().all()
assert "Review" not in actual_counts

# Critical target and measurement/selection decisions
def value_for(df, variable, column):
    rows = df.loc[df["Variable"] == variable, column]
    assert len(rows) == 1, f"Expected one row for {variable}"
    return rows.iloc[0]

assert value_for(feature_check, "Q49", "Selection_decision") == "Exclude"
assert value_for(feature_check, "Q192", "Selection_decision") == "Keep"
assert value_for(feature_check, "Q269", "Selection_decision") == "Keep"
assert value_for(feature_check, "Q272", "Selection_decision") == "Optional"
assert value_for(feature_check, "Q289", "Selection_decision") == "Optional"
assert value_for(feature_check, "Q290", "Selection_decision") == "Exclude"

for variable in ["Q270", "Q274"]:
    assert value_for(feature_check, variable, "Measurement_type") == "Numeric"
    assert value_for(dictionary_check, variable, "Measurement_type") == "Numeric"

assert value_for(feature_check, "Q284", "Measurement_type") == "Nominal"
assert value_for(dictionary_check, "Q284", "Measurement_type") == "Nominal"
assert value_for(feature_check, "Q254", "Measurement_type") == "Nominal"
assert value_for(dictionary_check, "Q254", "Measurement_type") == "Nominal"

for variable in ["Q294A", "Q294B"]:
    assert value_for(feature_check, variable, "Measurement_type") == "Ordinal"
    assert value_for(dictionary_check, variable, "Measurement_type") == "Ordinal"

# Diagnose CSV round-trip differences

def compare_dataframes(csv_df, memory_df, name):
    csv_df = csv_df.reset_index(drop=True)
    memory_df = memory_df.reset_index(drop=True)

    print(f"\n--- {name} ---")
    print("CSV shape:", csv_df.shape)
    print("Memory shape:", memory_df.shape)

    print("Same columns:", list(csv_df.columns) == list(memory_df.columns))

    if list(csv_df.columns) != list(memory_df.columns):
        print("CSV columns:", list(csv_df.columns))
        print("Memory columns:", list(memory_df.columns))
        return

    differences = []

    for column in csv_df.columns:
        left = csv_df[column]
        right = memory_df[column]

        equal = left.eq(right) | (
            left.isna() & right.isna()
        )

        mismatch = ~equal

        if mismatch.any():
            differences.append(column)

            print(f"\nColumn: {column}")
            print("Mismatch count:", mismatch.sum())

            display(
                pd.DataFrame({
                    "CSV value": left[mismatch].head(5),
                    "Memory value": right[mismatch].head(5)
                })
            )

    if not differences:
        print("All values match.")
        
feature_check["Selection_reason"] = (
    feature_check["Selection_reason"].fillna("")
)

feature_selection["Selection_reason"] = (
    feature_selection["Selection_reason"].fillna("")
)

compare_dataframes(
    feature_check,
    feature_selection,
    "Feature Selection"
)

compare_dataframes(
    dictionary_check,
    master_dictionary,
    "Master Variable Dictionary"
)

print("VALIDATION PASSED")
print("Feature Selection:", feature_check.shape)
print("Master Variable Dictionary:", dictionary_check.shape)
print("Selection counts:", actual_counts)
print("Q49 excluded; critical variable updates verified.")



--- Feature Selection ---
CSV shape: (372, 20)
Memory shape: (372, 20)
Same columns: True
All values match.

--- Master Variable Dictionary ---
CSV shape: (372, 18)
Memory shape: (372, 18)
Same columns: True
All values match.
VALIDATION PASSED
Feature Selection: (372, 20)
Master Variable Dictionary: (372, 18)
Selection counts: {'Keep': 248, 'Optional': 102, 'Exclude': 22}
Q49 excluded; critical variable updates verified.
